# Financial Ratio Analysis & DuPont Decomposition

*CFA Level 1 — Financial Statement Analysis*

This notebook develops a **from-scratch** framework for evaluating company performance through financial ratios and the DuPont decomposition of Return on Equity. We construct all metrics from raw financial-statement data using only NumPy, derive every formula algebraically, and visualise the results with Matplotlib.

**Learning objectives**

| # | Objective |
|---|-----------|
| 1 | Understand **why** ratio analysis removes scale effects and enables cross-sectional comparison. |
| 2 | Compute and interpret the five major ratio families: profitability, activity, liquidity, solvency, and valuation. |
| 3 | Decompose ROE into **3-factor** and **5-factor** DuPont trees. |
| 4 | Build radar charts, tornado charts, and multi-company dashboards entirely from scratch. |

**Prerequisites:**

- Familiarity with the three core financial statements (income statement, balance sheet, cash flow statement).
- Basic understanding of accounting terminology (revenue, COGS, EBIT, EBT, net income, assets, liabilities, equity).
- Comfort with Python and NumPy for numerical computation.

**Notation conventions used throughout this notebook:**

| Symbol | Meaning |
|--------|---------|
| NI | Net Income |
| Rev | Revenue (Sales) |
| EBIT | Earnings Before Interest and Taxes (Operating Income) |
| EBT | Earnings Before Taxes (Pre-tax Income) |
| COGS | Cost of Goods Sold |
| PP&E | Property, Plant and Equipment |
| SG&A | Selling, General and Administrative Expenses |
| D/E | Debt-to-Equity Ratio |
| D/A | Debt-to-Assets Ratio |
| ROA | Return on Assets |
| ROE | Return on Equity |

> **Key Concept:** Throughout this notebook, we use end-of-period balance-sheet values (rather than averages) for simplicity.  In practice, using average values is more accurate when balance-sheet items change significantly within the period.

---

## 1. Why Ratio Analysis?

### 1.1 Removing Scale Effects

Raw financial-statement numbers are difficult to compare across companies of different sizes.  A firm earning \$10 billion in revenue and \$1 billion in net income is not directly comparable with a firm earning \$500 million in revenue and \$75 million in net income.

**Ratio analysis** normalises financial data by expressing one line item as a fraction of another, producing **dimensionless** (or consistently dimensioned) metrics that allow:

- **Cross-sectional comparison** — benchmarking against peers within the same industry.
- **Time-series comparison** — tracking a single company's trajectory over multiple periods.
- **Standard-setting** — establishing minimum thresholds (e.g., loan covenants requiring a current ratio above 1.5).

> **Key Concept:** Ratios convert absolute dollar figures into relative measures, enabling apples-to-apples comparisons between a \$50 billion conglomerate and a \$500 million niche player.  Without ratios, scale differences dominate every comparison and mask true operational performance.

#### Why Ratios Matter in Practice

Financial analysts, credit-rating agencies, and portfolio managers use ratios as the **primary language** of fundamental analysis.  When a credit analyst at Moody's evaluates a bond issuer, they examine dozens of ratios — interest coverage, debt-to-EBITDA, free-cash-flow-to-debt — to assess default risk.  When an equity analyst at a buy-side fund screens for investment ideas, they filter by P/E, ROE, and revenue growth rates.

Ratios also form the backbone of **financial covenants** in loan agreements.  A typical bank loan might require:

| Covenant | Typical Threshold | Consequence of Breach |
|----------|-------------------|----------------------|
| Current Ratio | >= 1.5 | Technical default, potential acceleration |
| Debt-to-Equity | <= 2.0 | Restriction on additional borrowing |
| Interest Coverage | >= 3.0 | Lender may demand additional collateral |
| Minimum Net Worth | Varies | Triggers renegotiation of loan terms |

Breaching a covenant can trigger technical default, giving the lender the right to demand immediate repayment.  This makes ratio analysis not just an academic exercise but a practical, high-stakes tool.

#### Types of Ratios by Construction

Ratios can be classified by how they combine financial-statement items:

| Type | Construction | Example | Interpretation |
|------|-------------|---------|----------------|
| **Coverage** | Flow / Flow | EBIT / Interest | Times operating income covers interest |
| **Return** | Flow / Stock | Net Income / Equity | Return generated on invested capital |
| **Turnover** | Flow / Stock | Revenue / Assets | How many times assets "turn over" |
| **Component** | Stock / Stock | Current Assets / Current Liabilities | Relative size of balance-sheet items |

> **CFA Exam Tip:** Understanding the type of ratio helps with interpretation.  Coverage ratios measure cushion (higher = safer).  Return ratios measure performance (higher = better, usually).  Turnover ratios measure velocity (optimal level varies by industry).

#### The Ratio Analysis Process

A systematic approach to ratio analysis follows four steps:

1. **Compute** the ratios from audited financial statements.
2. **Compare** them against benchmarks — industry medians, historical trends, or covenant thresholds.
3. **Investigate** any significant deviations from benchmarks to understand the underlying cause.
4. **Conclude** with an overall assessment of the firm's financial health and trajectory.

> **CFA Exam Tip:** The CFA curriculum frames ratio analysis as part of a broader analytical process that includes common-size analysis, trend analysis, and regression analysis.  Ratios are the workhorse, but they are most powerful when combined with qualitative context (management quality, competitive position, regulatory environment).

### 1.2 The Five Families of Ratios

Financial ratios are conventionally grouped into five families:

| Family | Question Answered | Key Ratios |
|--------|-------------------|------------|
| **Profitability** | How effectively does the firm convert revenue into profit? | Gross margin, operating margin, net margin, ROA, ROE |
| **Activity (efficiency)** | How effectively does the firm use its assets to generate revenue? | Receivables turnover, inventory turnover, asset turnover |
| **Liquidity** | Can the firm meet its short-term obligations? | Current ratio, quick ratio, cash ratio |
| **Solvency** | Can the firm meet its long-term obligations? | D/E, D/A, interest coverage, fixed charge coverage |
| **Valuation** | How does the market price the firm relative to fundamentals? | P/E, P/B, P/S, EV/EBITDA |

> **Key Concept:** These five families span the entire balance sheet and income statement.  Together they provide a 360-degree view of a company's financial health.

#### How the Families Interconnect

The five families are not independent — they interact in important ways:

- **Profitability and Activity:** A firm can improve profitability by becoming more efficient (higher asset turnover).  The DuPont decomposition formalises this link.
- **Liquidity and Solvency:** Short-term liquidity problems can escalate into long-term solvency crises if not addressed.  Conversely, a firm with strong solvency (low leverage, high coverage) can typically access credit markets to solve short-term liquidity needs.
- **Profitability and Valuation:** Higher profitability generally commands higher valuation multiples, but the relationship is moderated by growth expectations and risk.
- **Solvency and Profitability:** Leverage (a solvency measure) amplifies profitability (ROE) through the equity multiplier in the DuPont framework.

Understanding these interconnections is essential for holistic financial analysis — examining ratios in isolation misses the bigger picture.

> **CFA Exam Tip:** The CFA exam frequently tests the interconnections between ratio families.  A common question format presents a scenario where one ratio changes and asks you to predict the effect on other ratios.  For example: "If a firm issues debt to repurchase equity, what happens to D/E, ROE, and interest coverage?"  (Answer: D/E rises, ROE rises if ROA exceeds the after-tax cost of debt, and interest coverage falls.)

### 1.3 Limitations of Ratio Analysis

> **Common Mistake:** Treating ratios as absolute truths without understanding context.  A current ratio of 1.2 might be healthy for a grocery chain (fast inventory turnover) but dangerously low for a capital-goods manufacturer.

Key limitations include:

1. **Accounting policy differences** — FIFO vs. LIFO, capitalisation vs. expensing, lease treatment.  Under FIFO, inventory reflects more recent costs while COGS reflects older costs; under LIFO, the opposite holds.  This asymmetry can cause two otherwise identical firms to report dramatically different margins and turnover ratios.
2. **Industry heterogeneity** — ratios only make sense relative to the correct peer group.  Comparing a bank's D/E ratio with a software company's is meaningless.
3. **Seasonality** — balance-sheet snapshots on different dates may distort ratios.  A retailer's balance sheet on 31 December (post-holiday) looks very different from its balance sheet on 30 September.
4. **Historical cost** — asset values on the balance sheet may not reflect current market values, especially for long-lived assets like real estate or intellectual property.
5. **Earnings management** — management may use discretionary accruals to paint a rosier picture.  Techniques include channel stuffing (shipping product early to book revenue), cookie-jar reserves (over-reserving in good years to smooth earnings in bad years), and capitalising operating expenses.
6. **One-time items** — restructuring charges, asset write-downs, and litigation settlements can distort single-period ratios.  Always examine whether unusual items are truly non-recurring.
7. **Conglomerate problem** — diversified firms operate in multiple industries, making peer selection difficult.  General Electric historically operated in aviation, healthcare, energy, and financial services — no single industry benchmark applies.
8. **Window dressing** — firms may temporarily improve balance-sheet ratios near reporting dates.  For example, a company might pay down short-term debt just before quarter-end to boost its current ratio, then re-borrow immediately after.
9. **Inflation effects** — in periods of high inflation, historical-cost accounting understates asset values and overstates profitability because depreciation is based on lower historical costs.
10. **Non-comparable fiscal years** — companies with different fiscal year-ends may be at different points in the business cycle when their statements are published.

> **CFA Exam Tip:** The CFA curriculum emphasises that ratio analysis is a *starting point* for analysis, not an end in itself.  Always seek the economic story behind the numbers.

#### Mitigating the Limitations

Analysts can address some of these limitations through careful preparation:

- **Adjusting financial statements** before computing ratios (e.g., converting LIFO to FIFO, capitalising operating leases, removing one-time items).
- **Using multi-year averages** to smooth out seasonality and one-time effects.
- **Applying common-size analysis** alongside ratio analysis for a richer picture.
- **Cross-referencing with cash-flow statements** to verify that reported earnings translate into actual cash.
- **Reading the footnotes** — the notes to the financial statements contain critical information about accounting policies, contingent liabilities, and off-balance-sheet items.

#### Common Accounting Adjustments for Ratio Comparability

| Issue | Adjustment | Effect on Ratios |
|-------|-----------|-----------------|
| LIFO vs. FIFO | Add LIFO reserve to inventory and equity | Increases current ratio, reduces inventory turnover |
| Operating leases (pre-IFRS 16) | Capitalise leases on balance sheet | Increases D/E, reduces asset turnover |
| R&D expensing | Capitalise and amortise R&D | Increases assets, changes margins |
| Goodwill impairment | Remove goodwill from assets | Reduces total assets, increases turnover |
| One-time charges | Add back to earnings | Increases margins, coverage ratios |

> **Key Concept:** The best analysts do not blindly compute ratios — they *adjust* the underlying data first to ensure comparability, then compute ratios, and finally interpret them in context.  Adjusted ratios are more comparable across firms and more informative for decision-making.

---

## 2. Setup

### 2.0 Environment and Dependencies

This notebook uses only core scientific Python libraries — no financial-analysis packages or black-box implementations:

- **NumPy** — for array-based computation of all financial ratios.
- **Matplotlib** — for all visualisations (bar charts, radar charts, tornado charts, small-multiples).
- **pandas** (display only) — for formatted table output.

> **Key Concept:** By building every ratio and visualisation from scratch, we ensure complete transparency in every calculation.  There are no hidden assumptions or library defaults — every formula is explicit and verifiable.

The synthetic data is designed to be internally consistent so that you can verify any computed ratio by hand using the raw financial-statement arrays.  This is an essential skill for the CFA exam, where you must be comfortable computing ratios from presented data without calculator shortcuts or software.

#### Computational Approach

Our implementation follows a consistent pattern for each ratio family:

1. **Define the ratio formula** as a vectorised NumPy operation over the 5-year time series.
2. **Compute the ratio** for all three companies simultaneously using array broadcasting.
3. **Visualise** the results using grouped bar charts for cross-sectional comparison and line charts for time-series analysis.
4. **Interpret** the results in the context of each company's profile and industry norms.

> **CFA Exam Tip:** On the exam, you will need to compute ratios by hand from given financial statements.  Practice computing each ratio family until the formulas are second nature.  Speed matters — the CFA Level 1 exam has 180 questions to answer in approximately 4.5 hours.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Tolerances
ATOL = 1e-8
RTOL = 1e-6

# Colour palette
PRIMARY   = "steelblue"
SECONDARY = "coral"
TERTIARY  = "seagreen"
ACCENT    = "gold"

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Setup complete.")

### 2.1 Synthetic Financial-Statement Data

We create **three synthetic companies** — AlphaCorp, BetaInc, and GammaTech — each with five years (Year 1 through Year 5) of income-statement, balance-sheet, and market data stored as NumPy arrays.

All figures are in **millions of dollars**.

| Company | Profile | Revenue Range | Leverage | Expected Characteristics |
|---------|---------|---------------|----------|------------------------|
| **AlphaCorp** | Mature industrial | ~5,000-5,600 | Moderate | Stable margins, moderate turnover, balanced DuPont |
| **BetaInc** | High-growth tech | ~2,000-5,200 | Low | High margins, lower turnover (heavy asset investment), low leverage |
| **GammaTech** | Utility (stable) | ~8,000-8,300 | High | Lower margins, low turnover (capital-intensive), high leverage |

> **Key Concept:** In practice you would pull these numbers from SEC filings (10-K / 10-Q) or a financial-data provider.  Here we generate plausible, internally consistent data so that every downstream ratio can be verified by hand.

#### Financial Statement Relationships

The synthetic data respects the fundamental accounting identity and income-statement cascade:

**Balance sheet identity:**
$$
\text{Total Assets} = \text{Total Liabilities} + \text{Shareholders' Equity}
$$

**Income statement cascade:**
$$
\text{Revenue} \xrightarrow{-\text{COGS}} \text{Gross Profit} \xrightarrow{-\text{SG\&A, D\&A}} \text{EBIT} \xrightarrow{-\text{Interest}} \text{EBT} \xrightarrow{-\text{Tax}} \text{Net Income}
$$

Each line item feeds into specific ratios, creating a complete analytical framework from a consistent set of financial data.

> **Common Mistake:** Using inconsistent data — for example, using beginning-of-period equity with end-of-period net income.  Our synthetic data uses period-end values throughout for internal consistency.

#### Data Dictionary

| Variable | Description | Statement | Used In |
|----------|-------------|-----------|---------|
| Revenue | Total sales | Income Statement | Margins, turnover, P/S |
| COGS | Cost of goods sold | Income Statement | Gross margin, inventory turnover |
| EBIT | Earnings before interest and taxes | Income Statement | Operating margin, interest coverage |
| Interest Expense | Cost of debt financing | Income Statement | Interest coverage, interest burden |
| EBT | Earnings before taxes | Income Statement | Tax burden |
| Net Income | Bottom-line profit | Income Statement | All profitability ratios |
| Total Assets | All assets | Balance Sheet | ROA, asset turnover, D/A |
| Equity | Shareholders' equity | Balance Sheet | ROE, D/E, equity multiplier |
| Current Assets | Short-term assets | Balance Sheet | Liquidity ratios |
| Current Liabilities | Short-term obligations | Balance Sheet | Liquidity ratios |

In [ ]:
# ---------------------------------------------------------------
# Synthetic 3-statement data for 3 companies x 5 years
# All figures in $millions
# ---------------------------------------------------------------
years = np.arange(1, 6)  # Year 1 .. Year 5
n_years = len(years)

# === AlphaCorp (mature industrial, moderate leverage) ===
alpha = {}
alpha["name"] = "AlphaCorp"
alpha["revenue"]          = np.array([5000, 5200, 5500, 5350, 5600])
alpha["cogs"]             = np.array([3200, 3380, 3520, 3450, 3580])
alpha["sga"]              = np.array([ 600,  620,  650,  640,  660])
alpha["depreciation"]     = np.array([ 250,  260,  270,  265,  275])
alpha["interest_expense"] = np.array([  80,   78,   75,   77,   74])
alpha["tax_rate"]         = np.array([0.25, 0.25, 0.25, 0.25, 0.25])

# Derived income-statement items
alpha["gross_profit"]     = alpha["revenue"] - alpha["cogs"]
alpha["operating_income"] = alpha["gross_profit"] - alpha["sga"] - alpha["depreciation"]
alpha["ebt"]              = alpha["operating_income"] - alpha["interest_expense"]
alpha["tax"]              = alpha["ebt"] * alpha["tax_rate"]
alpha["net_income"]       = alpha["ebt"] - alpha["tax"]
alpha["ebitda"]           = alpha["operating_income"] + alpha["depreciation"]

# Balance-sheet items (end of period)
alpha["cash"]              = np.array([ 300,  320,  350,  330,  360])
alpha["receivables"]       = np.array([ 450,  470,  500,  480,  510])
alpha["inventory"]         = np.array([ 600,  620,  640,  610,  650])
alpha["other_current"]     = np.array([  50,   55,   60,   58,   62])
alpha["current_assets"]    = alpha["cash"] + alpha["receivables"] + alpha["inventory"] + alpha["other_current"]
alpha["ppe_net"]           = np.array([3000, 3100, 3200, 3150, 3250])
alpha["total_assets"]      = alpha["current_assets"] + alpha["ppe_net"]
alpha["current_liab"]      = np.array([ 800,  830,  860,  840,  870])
alpha["long_term_debt"]    = np.array([1200, 1150, 1100, 1120, 1080])
alpha["total_liab"]        = alpha["current_liab"] + alpha["long_term_debt"]
alpha["equity"]            = alpha["total_assets"] - alpha["total_liab"]

# Market data
alpha["shares_outstanding"] = np.array([100, 100, 100, 100, 100])  # millions
alpha["share_price"]        = np.array([ 32,  34,  37,  35,  39])
alpha["market_cap"]         = alpha["shares_outstanding"] * alpha["share_price"]

# Fixed charges (lease payments)
alpha["lease_payments"]     = np.array([40, 42, 44, 43, 45])

# Daily operating expenses (for defensive interval)
alpha["daily_op_exp"]       = (alpha["cogs"] + alpha["sga"]) / 365.0

# === BetaInc (high-growth tech, low leverage) ===
beta = {}
beta["name"] = "BetaInc"
beta["revenue"]          = np.array([2000, 2600, 3400, 4200, 5200])
beta["cogs"]             = np.array([ 600,  780, 1020, 1260, 1560])
beta["sga"]              = np.array([ 700,  850, 1050, 1200, 1400])
beta["depreciation"]     = np.array([ 100,  120,  150,  180,  210])
beta["interest_expense"] = np.array([  20,   22,   25,   28,   30])
beta["tax_rate"]         = np.array([0.22, 0.22, 0.22, 0.22, 0.22])

beta["gross_profit"]     = beta["revenue"] - beta["cogs"]
beta["operating_income"] = beta["gross_profit"] - beta["sga"] - beta["depreciation"]
beta["ebt"]              = beta["operating_income"] - beta["interest_expense"]
beta["tax"]              = beta["ebt"] * beta["tax_rate"]
beta["net_income"]       = beta["ebt"] - beta["tax"]
beta["ebitda"]           = beta["operating_income"] + beta["depreciation"]

beta["cash"]              = np.array([ 800,  900, 1100, 1300, 1600])
beta["receivables"]       = np.array([ 200,  260,  340,  420,  520])
beta["inventory"]         = np.array([  80,  100,  130,  160,  200])
beta["other_current"]     = np.array([  20,   25,   30,   35,   40])
beta["current_assets"]    = beta["cash"] + beta["receivables"] + beta["inventory"] + beta["other_current"]
beta["ppe_net"]           = np.array([ 800, 1000, 1300, 1600, 2000])
beta["total_assets"]      = beta["current_assets"] + beta["ppe_net"]
beta["current_liab"]      = np.array([ 300,  370,  460,  550,  660])
beta["long_term_debt"]    = np.array([ 200,  220,  250,  280,  300])
beta["total_liab"]        = beta["current_liab"] + beta["long_term_debt"]
beta["equity"]            = beta["total_assets"] - beta["total_liab"]

beta["shares_outstanding"] = np.array([200, 200, 200, 200, 200])
beta["share_price"]        = np.array([ 18,  24,  33,  42,  55])
beta["market_cap"]         = beta["shares_outstanding"] * beta["share_price"]

beta["lease_payments"]     = np.array([15, 18, 22, 26, 30])
beta["daily_op_exp"]       = (beta["cogs"] + beta["sga"]) / 365.0

# === GammaTech (utility, high leverage, stable) ===
gamma = {}
gamma["name"] = "GammaTech"
gamma["revenue"]          = np.array([8000, 8100, 8200, 8150, 8300])
gamma["cogs"]             = np.array([5200, 5300, 5350, 5320, 5400])
gamma["sga"]              = np.array([ 800,  810,  820,  815,  830])
gamma["depreciation"]     = np.array([ 500,  510,  520,  515,  525])
gamma["interest_expense"] = np.array([ 300,  295,  290,  292,  285])
gamma["tax_rate"]         = np.array([0.28, 0.28, 0.28, 0.28, 0.28])

gamma["gross_profit"]     = gamma["revenue"] - gamma["cogs"]
gamma["operating_income"] = gamma["gross_profit"] - gamma["sga"] - gamma["depreciation"]
gamma["ebt"]              = gamma["operating_income"] - gamma["interest_expense"]
gamma["tax"]              = gamma["ebt"] * gamma["tax_rate"]
gamma["net_income"]       = gamma["ebt"] - gamma["tax"]
gamma["ebitda"]           = gamma["operating_income"] + gamma["depreciation"]

gamma["cash"]              = np.array([ 400,  410,  420,  415,  430])
gamma["receivables"]       = np.array([ 700,  710,  720,  715,  730])
gamma["inventory"]         = np.array([ 900,  910,  920,  915,  930])
gamma["other_current"]     = np.array([  80,   82,   85,   83,   88])
gamma["current_assets"]    = gamma["cash"] + gamma["receivables"] + gamma["inventory"] + gamma["other_current"]
gamma["ppe_net"]           = np.array([6000, 6050, 6100, 6080, 6150])
gamma["total_assets"]      = gamma["current_assets"] + gamma["ppe_net"]
gamma["current_liab"]      = np.array([1200, 1210, 1220, 1215, 1230])
gamma["long_term_debt"]    = np.array([3500, 3450, 3400, 3420, 3380])
gamma["total_liab"]        = gamma["current_liab"] + gamma["long_term_debt"]
gamma["equity"]            = gamma["total_assets"] - gamma["total_liab"]

gamma["shares_outstanding"] = np.array([500, 500, 500, 500, 500])
gamma["share_price"]        = np.array([ 15,  15,  16,  15,  16])
gamma["market_cap"]         = gamma["shares_outstanding"] * gamma["share_price"]

gamma["lease_payments"]     = np.array([60, 62, 64, 63, 65])
gamma["daily_op_exp"]       = (gamma["cogs"] + gamma["sga"]) / 365.0

companies = [alpha, beta, gamma]
company_names = [c["name"] for c in companies]

print("Synthetic data created for:", ", ".join(company_names))
print(f"Each company has {n_years} years of data (Year 1 to Year 5).")

## 3. Profitability Ratios

Profitability ratios measure a firm's ability to generate profit relative to revenue, assets, or equity.  They answer the fundamental question: *"How effectively does the firm convert inputs into bottom-line returns?"*

Profitability can be measured at multiple levels of the income statement, each providing a different perspective:

| Level | Metric | What It Captures | Key Comparison Use |
|-------|--------|------------------|--------------------|
| Gross | Gross Margin | Pricing power and production cost efficiency | Within-industry |
| Operating | Operating Margin | Core business profitability (includes SG&A) | Cross-industry (capital-structure neutral) |
| Pre-tax | Pre-tax Margin | Profitability after financing costs, before tax | Cross-border (tax-neutral) |
| Net | Net Margin | Bottom-line profitability after all costs | Same-industry, same-country |
| Asset-based | ROA | Return generated per dollar of total assets | Across capital structures |
| Equity-based | ROE | Return earned on shareholders' capital | Equity investors' perspective |

> **Key Concept:** As you move down the income statement from gross profit to net income, each margin incorporates additional cost categories.  A firm with strong gross margins but weak net margins is likely burdened by high operating expenses, heavy interest costs, or a high tax rate — each pointing to a different strategic issue.

#### Industry Benchmarks for Profitability

Profitability ratios vary enormously across industries.  Understanding typical ranges prevents misinterpretation:

| Industry | Typical Gross Margin | Typical Operating Margin | Typical Net Margin | Typical ROE |
|----------|---------------------|------------------------|--------------------|-------------|
| Software / SaaS | 70 - 85% | 20 - 35% | 15 - 30% | 20 - 40% |
| Pharmaceuticals | 60 - 80% | 20 - 30% | 15 - 25% | 15 - 30% |
| Retail (grocery) | 25 - 35% | 3 - 5% | 1 - 3% | 10 - 20% |
| Utilities | 30 - 50% | 15 - 25% | 8 - 15% | 8 - 12% |
| Banking | N/A (use NIM) | N/A (use efficiency ratio) | 20 - 35% | 10 - 15% |
| Automotive | 15 - 25% | 5 - 10% | 3 - 7% | 10 - 20% |

> **CFA Exam Tip:** Always compare profitability ratios within the same industry.  Comparing a software firm's 25% net margin with a grocery chain's 2% net margin tells you nothing about which is better managed — it only reflects structural differences in their business models.

#### The Profitability-Growth Trade-off

Not all firms maximise current profitability.  High-growth firms often sacrifice near-term margins to invest in market share:

- **Amazon** operated at near-zero net margins for years while building its logistics and cloud infrastructure.
- **Tesla** was unprofitable for over a decade while investing in manufacturing scale.
- **Pharmaceutical companies** show depressed margins during heavy R&D investment phases.

> **Common Mistake:** Penalising a firm for low current margins without examining whether those margins reflect deliberate investment in future growth.  The correct approach is to assess whether the investments are likely to generate adequate returns over time.

### 3.1 Gross Profit Margin

$$
\text{Gross Margin} = \frac{\text{Revenue} - \text{COGS}}{\text{Revenue}} = \frac{\text{Gross Profit}}{\text{Revenue}}
$$

**Interpretation:** The percentage of each revenue dollar remaining after paying for the direct cost of goods sold.  A high gross margin indicates pricing power or low production costs.

> **Key Concept:** Gross margin is the *first* profitability gate.  If gross margin is thin, there is little room for operating expenses, interest, and taxes to be covered downstream.  Think of it as the raw economic value the firm creates before any overhead costs.

**What drives gross margin?**

- **Pricing power:** Brands with strong customer loyalty (Apple, luxury goods) can charge premium prices, leading to gross margins of 40-70%.
- **Cost of inputs:** Commodity-dependent firms (steel, agriculture) face volatile COGS that compress margins unpredictably.
- **Production efficiency:** Economies of scale reduce per-unit costs as volume increases — this is why high-volume manufacturers often show improving gross margins over time.
- **Product mix:** Shifting sales toward higher-margin products (e.g., from hardware to software) improves overall gross margin.
- **Vertical integration:** Firms that control their supply chain (e.g., owning raw material sources) can reduce COGS and boost margins.

**Gross margin trend analysis:**

| Trend | Signal | Possible Causes |
|-------|--------|-----------------|
| Expanding gross margin | Positive | Pricing power, cost reductions, favourable product mix |
| Stable gross margin | Neutral | Balanced cost and pricing dynamics |
| Compressing gross margin | Negative | Input cost inflation, competitive pricing pressure, unfavourable mix |
| Volatile gross margin | Warning | Commodity exposure, inconsistent pricing strategy |

> **Common Mistake:** Ignoring the effect of accounting methods on gross margin.  Under FIFO, rising input costs make COGS appear lower (using older, cheaper inventory), inflating gross margin.  Under LIFO, the opposite occurs.  Always check the inventory accounting method before comparing gross margins across firms.

### 3.2 Operating Profit Margin

$$
\text{Operating Margin} = \frac{\text{Operating Income (EBIT)}}{\text{Revenue}}
$$

**Interpretation:** Captures both production efficiency **and** control of operating expenses (SG&A, depreciation).  It strips out financing decisions and tax effects, making it ideal for comparing firms with different capital structures.

**The gross-to-operating margin bridge** reveals operational efficiency:

$$
\text{Operating Margin} = \text{Gross Margin} - \frac{\text{SG\&A} + \text{D\&A}}{\text{Revenue}}
$$

A firm with 70% gross margin but only 15% operating margin is spending 55% of revenue on operating costs — which may be justified (e.g., heavy R&D in pharma) or may indicate bloated overhead.

> **CFA Exam Tip:** Operating margin is often called "EBIT margin" in practice.  Some analysts prefer EBITDA margin (adding back depreciation and amortisation) for capital-intensive firms to remove the distortion of different depreciation policies.  The CFA curriculum uses both — know the difference.

### 3.3 Net Profit Margin

$$
\text{Net Margin} = \frac{\text{Net Income}}{\text{Revenue}}
$$

**Interpretation:** The *bottom-line* percentage.  Includes all costs — operating, financing, and tax.  Useful as a comprehensive profitability gauge, but sensitive to capital-structure and tax-regime differences.

**The margin waterfall** from gross to net shows where profit is consumed:

| Stage | Formula | What Reduces It |
|-------|---------|-----------------|
| Gross Margin | (Revenue - COGS) / Revenue | Direct production costs |
| Operating Margin | EBIT / Revenue | SG&A, R&D, depreciation |
| Pre-tax Margin | EBT / Revenue | Interest expense |
| Net Margin | NI / Revenue | Income taxes |

> **Common Mistake:** Comparing net margins across firms with very different capital structures.  A highly leveraged firm pays more interest, which reduces net margin even if its operations are equally efficient.  Use operating margin for fairer operational comparisons.

### 3.4 Return on Assets (ROA)

$$
\text{ROA} = \frac{\text{Net Income}}{\text{Total Assets}}
$$

**Interpretation:** How much profit the firm generates per dollar of assets, regardless of how those assets are financed.

ROA can also be decomposed using DuPont logic:

$$
\text{ROA} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} = \text{Net Margin} \times \text{Asset Turnover}
$$

This two-factor decomposition reveals whether high ROA comes from profitability (high margins) or efficiency (high turnover).  As we will see in Section 8, the 3-factor DuPont simply adds leverage on top of this ROA decomposition.

> **CFA Exam Tip:** Some textbooks define ROA using *average* total assets to smooth out intra-year balance-sheet fluctuations.  The CFA curriculum accepts both end-of-period and average formulations — just be consistent.

### 3.5 Return on Equity (ROE)

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Shareholders' Equity}}
$$

**Interpretation:** The return earned on owners' capital.  ROE is the single most important ratio for equity investors and is the ratio that the DuPont decomposition dissects.

**The relationship between ROA and ROE:**

$$
\text{ROE} = \text{ROA} \times \frac{\text{Total Assets}}{\text{Equity}} = \text{ROA} \times \text{Equity Multiplier}
$$

This means ROE = ROA when the firm has no debt (equity multiplier = 1).  As the firm takes on debt, the equity multiplier exceeds 1, and ROE exceeds ROA.  The spread between ROE and ROA is a direct measure of the leverage amplification effect.

> **Common Mistake:** A very high ROE can be driven by excessive leverage rather than genuine operating efficiency.  Always check whether high ROE comes from high margins or high leverage — the DuPont framework (Section 8) addresses exactly this.

> **Key Concept:** The hierarchy of profitability ratios forms a cascade: Gross Margin tells you about production, Operating Margin tells you about the whole business operation, Net Margin tells you about the bottom line after all stakeholders (creditors, government), and ROE tells you about the return to the residual claimant (shareholders).  Each step down incorporates more factors and is influenced by more decisions.

In [ ]:
# ---------------------------------------------------------------
# 3. Profitability Ratios — compute for all companies
# ---------------------------------------------------------------

def profitability_ratios(c):
    """Return dict of profitability ratios for a company dict."""
    return {
        "Gross Margin":     c["gross_profit"] / c["revenue"],
        "Operating Margin": c["operating_income"] / c["revenue"],
        "Net Margin":       c["net_income"] / c["revenue"],
        "ROA":              c["net_income"] / c["total_assets"],
        "ROE":              c["net_income"] / c["equity"],
    }

# Compute and display
for c in companies:
    ratios = profitability_ratios(c)
    print(f"\n{'='*55}")
    print(f"  {c['name']} — Profitability Ratios")
    print(f"{'='*55}")
    print(f"{'Ratio':<22}", end="")
    for y in years:
        print(f"  Yr{y:d}", end="")
    print()
    print("-" * 55)
    for name, vals in ratios.items():
        print(f"{name:<22}", end="")
        for v in vals:
            print(f" {v:6.1%}", end="")
        print()

#### Profitability Ratios: Interpretation of Results

The profitability charts above reveal critical differences among our three companies.  When analysing the output:

- **Gross margin differences** reflect business-model fundamentals — high-margin firms typically have pricing power, proprietary technology, or brand premium.
- **The gross-to-operating margin drop** reveals the burden of operating expenses.  A firm with 60% gross margin but only 15% operating margin spends heavily on SG&A or R&D.
- **The operating-to-net margin drop** captures financing costs and taxes.  A large gap here signals heavy leverage or an unfavourable tax position.
- **ROA vs. ROE divergence** indicates leverage.  When ROE substantially exceeds ROA, the firm is using significant financial leverage to amplify equity returns.

> **Common Mistake:** Concluding that a firm with higher margins is "better managed" without considering industry context.  A software company with 20% net margin may actually be underperforming relative to peers averaging 25%, while a grocery chain with 2% net margin may be a best-in-class operator.

> **CFA Exam Tip:** When presented with profitability data for multiple firms, the exam often asks you to explain *why* margins differ.  The answer typically involves one or more of: industry structure, competitive position, cost management, capital structure, and tax jurisdiction.

#### Profitability Ratio Diagnostic Framework

When you observe profitability ratios, systematically diagnose the results:

| Observation | Possible Causes | Follow-up Analysis |
|-------------|-----------------|-------------------|
| High gross margin, low operating margin | Heavy SG&A or R&D spending | Check if R&D is creating future value |
| Declining gross margin over time | Input cost inflation, pricing pressure, adverse product mix | Examine COGS components, pricing strategy |
| ROE >> ROA | High financial leverage | Check D/E ratio, interest coverage |
| ROA declining while ROE stable | Increasing leverage masking operational decline | Perform DuPont decomposition |
| Net margin volatile but operating margin stable | Fluctuating interest rates or tax changes | Examine interest expense and effective tax rate trends |

> **Key Concept:** Profitability ratios should always be examined as a *set*, not individually.  The pattern across gross, operating, and net margins reveals the story of where value is created and where it is consumed as you move down the income statement.

In [ ]:
# ---------------------------------------------------------------
# Visualise: Net Margin trend for all 3 companies
# ---------------------------------------------------------------
fig, ax = plt.subplots()
colours = [PRIMARY, SECONDARY, TERTIARY]
for c, col in zip(companies, colours):
    margin = c["net_income"] / c["revenue"]
    ax.plot(years, margin * 100, marker="o", color=col, linewidth=2, label=c["name"])

ax.set_xlabel("Year")
ax.set_ylabel("Net Profit Margin (%)")
ax.set_title("Net Profit Margin — 5-Year Trend")
ax.set_xticks(years)
ax.legend()
plt.tight_layout()
plt.show()

#### Additional Profitability Analysis

The second profitability chart above extends the analysis with ROA and ROE metrics.  Key comparisons:

- **ROA spread across companies:** A wide spread in ROA across companies operating in the same industry suggests genuine differences in operational efficiency.  If the spread is narrow, the industry may be highly competitive with limited room for differentiation.
- **ROE vs. ROA gap as leverage indicator:** The mathematical relationship $\text{ROE} = \text{ROA} \times \text{Equity Multiplier}$ means:
  - If ROE ≈ ROA, the firm has minimal leverage.
  - If ROE >> ROA, the firm uses significant leverage to amplify returns.
  - If ROE < ROA (unusual), the firm is losing value through its capital structure (cost of debt exceeds ROA).

> **Key Concept:** A useful mental model is to think of ROA as the "operating return" and the gap between ROE and ROA as the "leverage bonus."  The DuPont framework in Section 8 will formalise this decomposition.

| Company Profile | Expected Pattern |
|----------------|-----------------|
| Conservative (low leverage) | ROE close to ROA, both moderate |
| Growth-oriented (moderate leverage) | ROE moderately above ROA |
| Aggressive (high leverage) | ROE much higher than ROA — but also more volatile |

> **CFA Exam Tip:** If ROA is below the firm's after-tax cost of debt, leverage actually *reduces* ROE rather than increasing it.  This is because the firm earns less on its assets than it pays on its debt — a situation called "negative financial leverage."  Check for this condition before concluding that higher leverage always boosts ROE.

#### The Leverage Amplification Visual Test

A quick visual test: plot ROA and ROE side by side for each company.  The vertical gap between the two bars represents the leverage amplification effect.  Companies where this gap is large are deriving a significant portion of their equity returns from debt — not from operational excellence.  This visual pattern is the single most important pre-DuPont diagnostic.

> **Key Concept:** If ROE is 20% and ROA is 18%, leverage contributes only 2 percentage points.  If ROE is 20% and ROA is 8%, leverage contributes 12 percentage points — the firm's equity returns are overwhelmingly driven by financial engineering rather than business performance.

## 4. Activity (Efficiency) Ratios

Activity ratios gauge how efficiently a firm deploys its assets to generate revenue.  They bridge the income statement and the balance sheet.

The core idea is simple: **assets should earn their keep**.  Every dollar invested in inventory, receivables, or fixed assets should contribute to revenue generation.  When assets sit idle or turn over slowly, they represent a drag on returns.

Activity ratios are closely linked to the **cash conversion cycle** — the time it takes for a firm to convert its investment in inventory and other resources into cash flows from sales:

$$
\text{Cash Conversion Cycle} = \text{DIH} + \text{DSO} - \text{Days Payable Outstanding}
$$

where DIH = Days Inventory on Hand, DSO = Days Sales Outstanding.

> **Key Concept:** A shorter cash conversion cycle means the firm recovers cash faster, reducing its need for external financing.  Dell famously achieved a *negative* cash conversion cycle by collecting from customers before paying suppliers — effectively using supplier financing to fund operations.

| Ratio | Numerator | Denominator | Interpretation |
|-------|-----------|-------------|----------------|
| Receivables Turnover | Revenue | Accounts Receivable | Collection speed |
| Inventory Turnover | COGS | Inventory | Inventory velocity |
| Total Asset Turnover | Revenue | Total Assets | Overall asset productivity |
| Fixed Asset Turnover | Revenue | Net PP&E | Capital asset productivity |
| Payables Turnover | Purchases (or COGS) | Accounts Payable | Payment speed to suppliers |

### 4.3 Total Asset Turnover

$$
\text{Total Asset Turnover} = \frac{\text{Revenue}}{\text{Total Assets}}
$$

**Interpretation:** Revenue generated per dollar of assets.  This ratio is a **key driver** in the DuPont decomposition.

A firm can improve asset turnover by either (a) growing revenue faster than assets, or (b) reducing assets while maintaining revenue (e.g., selling underutilised facilities, tightening working capital).

**Total asset turnover by business model:**

| Business Model | Typical Turnover | Explanation |
|----------------|-----------------|-------------|
| Retail / wholesale | 2.0 - 3.5x | High volume, relatively few assets |
| Light manufacturing | 1.0 - 2.0x | Moderate asset base |
| Heavy manufacturing | 0.5 - 1.0x | Large fixed asset base |
| Utilities | 0.2 - 0.5x | Massive infrastructure investments |
| Software | 0.5 - 1.5x | Low tangible assets but high intangibles |

### 4.4 Fixed Asset Turnover

$$
\text{Fixed Asset Turnover} = \frac{\text{Revenue}}{\text{Net PP\&E}}
$$

**Interpretation:** How efficiently the firm uses its long-lived tangible assets to generate sales.  Capital-intensive industries (utilities, manufacturing) tend to have lower fixed-asset turnover.

> **CFA Exam Tip:** When comparing firms, ensure consistency in accounting for leases — capitalised leases inflate PP&E and total assets, reducing turnover ratios.  Under IFRS 16, all leases with terms longer than 12 months must be capitalised, significantly affecting turnover ratios for firms with large operating lease portfolios (airlines, retailers).

> **Key Concept:** The relationship between asset turnover and profit margins is often *inverse* — capital-light businesses (consulting, software) have high turnover but may face margin pressure from low barriers to entry, while capital-heavy businesses (utilities, telecoms) have low turnover but enjoy regulated or oligopolistic margins.  This trade-off is visible in the DuPont decomposition.

#### The Asset Turnover Lifecycle

A firm's asset turnover typically follows a lifecycle pattern:

1. **Start-up phase:** Low turnover — assets are being built but revenue is minimal.
2. **Growth phase:** Rising turnover — revenue grows faster than the asset base.
3. **Maturity phase:** Stable turnover — revenue and assets grow in proportion.
4. **Decline phase:** Falling turnover — revenue declines but assets remain on the books.

Understanding where a firm sits in this lifecycle is essential for interpreting its turnover ratio.

> **CFA Exam Tip:** A sudden drop in asset turnover may indicate a large acquisition (assets jumped) or a revenue decline.  Always check the components — did the numerator (revenue) fall, or did the denominator (assets) rise?

### 4.3 Total Asset Turnover

$$
\text{Total Asset Turnover} = \frac{\text{Revenue}}{\text{Total Assets}}
$$

**Interpretation:** Revenue generated per dollar of assets.  This ratio is a **key driver** in the DuPont decomposition.

A firm can improve asset turnover by either (a) growing revenue faster than assets, or (b) reducing assets while maintaining revenue (e.g., selling underutilised facilities, tightening working capital).

### 4.4 Fixed Asset Turnover

$$
\text{Fixed Asset Turnover} = \frac{\text{Revenue}}{\text{Net PP\&E}}
$$

**Interpretation:** How efficiently the firm uses its long-lived tangible assets to generate sales.  Capital-intensive industries (utilities, manufacturing) tend to have lower fixed-asset turnover.

> **CFA Exam Tip:** When comparing firms, ensure consistency in accounting for leases — capitalised leases inflate PP&E and total assets, reducing turnover ratios.

> **Key Concept:** The relationship between asset turnover and profit margins is often *inverse* — capital-light businesses (consulting, software) have high turnover but may face margin pressure from low barriers to entry, while capital-heavy businesses (utilities, telecoms) have low turnover but enjoy regulated or oligopolistic margins.  This trade-off is visible in the DuPont decomposition.

#### Turnover Ratio Interpretation Checklist

When analysing turnover ratios, use this checklist to ensure comprehensive interpretation:

1. Compare to industry peers — is the firm above or below the peer median?
2. Examine the trend — is turnover improving or deteriorating?
3. Decompose the change — did the numerator (revenue) or denominator (assets) drive the change?
4. Check for one-time effects — did an acquisition, divestiture, or write-down distort the ratio?
5. Link to DuPont — how does this turnover level affect the firm's ROE decomposition?
6. Consider the business cycle — is the current period representative, or is it a cyclical peak/trough?
7. Cross-reference with margins — is the margin-turnover trade-off at play?

> **Key Concept:** Turnover ratios are most informative when analysed in conjunction with profitability ratios, because the margin-turnover trade-off means that changes in one often affect the other.

In [ ]:
# ---------------------------------------------------------------
# 4. Activity (Efficiency) Ratios
# ---------------------------------------------------------------

def activity_ratios(c):
    recv_turnover = c["revenue"] / c["receivables"]
    inv_turnover  = c["cogs"] / c["inventory"]
    return {
        "Receivables Turnover": recv_turnover,
        "DSO (days)":           365.0 / recv_turnover,
        "Inventory Turnover":   inv_turnover,
        "DIH (days)":           365.0 / inv_turnover,
        "Total Asset Turnover": c["revenue"] / c["total_assets"],
        "Fixed Asset Turnover": c["revenue"] / c["ppe_net"],
    }

for c in companies:
    ratios = activity_ratios(c)
    print(f"\n{'='*60}")
    print(f"  {c['name']} — Activity Ratios")
    print(f"{'='*60}")
    print(f"{'Ratio':<24}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 60)
    for name, vals in ratios.items():
        print(f"{name:<24}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

#### Activity Ratios: Interpretation of Results

The activity ratio charts above illustrate the efficiency profiles of our three companies.  Points to analyse:

- **Receivables turnover comparison:** Firms with significantly lower receivables turnover than peers may have more lenient credit terms (a competitive strategy to win customers) or may be struggling to collect from customers (a credit-quality issue).
- **Inventory turnover patterns:** Declining inventory turnover over time is a classic early warning signal.  It may indicate weakening demand, production inefficiencies, or product obsolescence.
- **Asset turnover as a DuPont input:** Remember that total asset turnover is one of the three DuPont factors.  When you observe a low asset turnover, it directly depresses ROE — unless compensated by higher margins or leverage.

> **Key Concept:** Activity ratios are the bridge between the balance sheet and the income statement.  They measure how effectively balance-sheet investments (assets) translate into income-statement results (revenue).  Improving activity ratios without sacrificing margins is the hallmark of excellent operational management.

> **CFA Exam Tip:** When computing receivables turnover, use end-of-period receivables (simpler) or average receivables (more accurate).  The CFA curriculum accepts both approaches — just be consistent and state your assumption.

#### The Cash Conversion Cycle in Practice

Combining the activity ratios yields the cash conversion cycle (CCC):

$$
\text{CCC} = \text{DIH} + \text{DSO} - \text{DPO}
$$

| Component | Effect of Increase | Effect on Cash Needs |
|-----------|--------------------|---------------------|
| DIH (inventory days) | Inventory sits longer | More cash tied up |
| DSO (receivables days) | Customers pay slower | More cash tied up |
| DPO (payables days) | Firm pays suppliers slower | Less cash needed |

A shorter CCC is generally better — it means the firm converts its investments into cash more quickly.  Companies like Amazon and Dell have famously achieved negative CCCs by collecting from customers before paying suppliers.

> **Common Mistake:** Ignoring the quality dimension of activity ratios.  A declining DSO might look positive (faster collection), but it could result from writing off uncollectible accounts rather than actually collecting faster.  Always cross-reference with the allowance for doubtful accounts.

In [ ]:
# ---------------------------------------------------------------
# Visualise: Total Asset Turnover comparison
# ---------------------------------------------------------------
fig, ax = plt.subplots()
width = 0.25
x = np.arange(n_years)

for i, (c, col) in enumerate(zip(companies, colours)):
    tat = c["revenue"] / c["total_assets"]
    ax.bar(x + i * width, tat, width, color=col, label=c["name"], edgecolor="white")

ax.set_xlabel("Year")
ax.set_ylabel("Total Asset Turnover (x)")
ax.set_title("Total Asset Turnover — Cross-Sectional Comparison")
ax.set_xticks(x + width)
ax.set_xticklabels([f"Yr {y}" for y in years])
ax.legend()
plt.tight_layout()
plt.show()

#### Additional Activity Analysis

The second set of activity charts extends the analysis to total and fixed asset turnover.  These ratios provide crucial insight into capital allocation effectiveness:

**Total asset turnover** is the single most important activity ratio because it directly feeds into the DuPont decomposition.  A firm with low total asset turnover must compensate with either high margins or high leverage to achieve competitive ROE.

**Fixed asset turnover trends** deserve special attention:

- **Declining fixed asset turnover** after a major capital expenditure is expected — new assets take time to generate revenue.  This is the "J-curve" effect of investment.
- **Persistently low fixed asset turnover** may indicate over-investment, poor capital allocation, or technological obsolescence.
- **Very high fixed asset turnover** may indicate the firm is under-investing in maintenance or expansion, borrowing capacity from the future.

> **Key Concept:** Capital-intensive firms (utilities, telecoms, manufacturing) typically show fixed asset turnover below 2.0x, while asset-light firms (consulting, software) may show turnover above 5.0x.  Always interpret in the industry context.

> **Common Mistake:** Comparing fixed asset turnover across firms with different asset ages.  A firm with fully depreciated (old) equipment will show higher turnover than a firm with newly purchased equipment — not because it is more efficient, but because its denominator (net PP&E) is smaller due to accumulated depreciation.

**Connecting activity ratios to cash flow:**

Activity ratios have direct cash-flow implications.  Improving receivables turnover accelerates cash collection; improving inventory turnover reduces cash tied up in stock.  The combined effect flows through to operating cash flow:

$$
\text{Operating CF} \approx \text{Net Income} + \text{Depreciation} - \Delta\text{Working Capital}
$$

When activity ratios improve (higher turnover = lower working capital needs), the change in working capital becomes favourable, boosting operating cash flow above reported net income.  This is why firms with improving activity ratios often surprise positively on cash-flow metrics.

## 5. Liquidity Ratios

Liquidity ratios assess whether a firm can meet its **short-term obligations** as they come due.

Liquidity is not the same as solvency.  A firm can be solvent (assets exceed liabilities) yet illiquid (unable to convert assets to cash quickly enough to pay bills).  The distinction matters because illiquidity can force bankruptcy even when the balance sheet is fundamentally healthy — a lesson painfully demonstrated during the 2008 financial crisis when several banks failed due to liquidity runs despite having positive net worth.

The standard liquidity ratios form a hierarchy from most inclusive to most conservative:

| Ratio | Numerator | Denominator | Stringency |
|-------|-----------|-------------|------------|
| Current Ratio | All current assets | Current liabilities | Least conservative |
| Quick Ratio | Cash + Receivables | Current liabilities | Moderate |
| Cash Ratio | Cash only | Current liabilities | Most conservative |
| Defensive Interval | Liquid assets | Daily operating expenses | Time-based measure |

> **CFA Exam Tip:** Know all four liquidity ratios and be able to explain when each is most appropriate.  The cash ratio is most relevant for firms in financial distress; the current ratio is the standard screening metric for credit analysis.

#### Why Multiple Liquidity Measures?

Different stakeholders care about different liquidity thresholds:

- **Trade creditors** (suppliers) want assurance that current assets cover current liabilities — the **current ratio** satisfies this need.
- **Short-term lenders** (commercial paper holders) want assurance that *liquid* assets cover near-term obligations — the **quick ratio** is more relevant.
- **Bondholders evaluating covenant compliance** need the specific metric stipulated in the bond indenture.
- **Management** monitors the **defensive interval** to understand operational runway without additional revenue.

> **Key Concept:** Liquidity is about *timing*, not just *amounts*.  A firm with current assets of \$100M and current liabilities of \$80M looks liquid (current ratio = 1.25), but if \$70M of those current assets is slow-moving inventory and \$60M of the liabilities mature next week, the firm has a severe liquidity crisis.  This is why the quick ratio and cash ratio exist — they strip out less liquid assets.

### 5.1 Current Ratio

$$
\text{Current Ratio} = \frac{\text{Current Assets}}{\text{Current Liabilities}}
$$

**Interpretation:** The broadest liquidity measure.  A ratio above 1.0 implies current assets exceed current liabilities.

**Benchmarks by industry:**

| Industry | Typical Current Ratio | Notes |
|----------|----------------------|-------|
| Technology | 2.0 - 4.0 | Large cash balances |
| Manufacturing | 1.5 - 2.5 | Moderate inventory |
| Retail | 1.0 - 1.5 | Fast-turning inventory, vendor financing |
| Utilities | 0.8 - 1.2 | Stable, predictable cash flows |

> **Common Mistake:** A very *high* current ratio is not always good — it may indicate excessive inventory or idle cash that is not being deployed productively.  The optimal current ratio depends on the industry's cash conversion cycle and the predictability of its cash flows.

### 5.2 Quick Ratio (Acid-Test)

$$
\text{Quick Ratio} = \frac{\text{Cash} + \text{Receivables}}{\text{Current Liabilities}}
$$

**Interpretation:** A stricter test that removes inventory (the least liquid current asset) from the numerator.  The quick ratio is particularly important for firms with slow-moving or perishable inventory.

The difference between the current ratio and quick ratio reveals the firm's dependence on inventory for liquidity coverage:

$$
\text{Current Ratio} - \text{Quick Ratio} \approx \frac{\text{Inventory}}{\text{Current Liabilities}}
$$

A large gap suggests heavy reliance on inventory, which may not be readily convertible to cash during a downturn.

> **CFA Exam Tip:** The quick ratio is sometimes called the "acid-test ratio" because it tests whether a firm can survive if its inventory becomes worthless — an acid test of financial resilience.

### 5.3 Cash Ratio

$$
\text{Cash Ratio} = \frac{\text{Cash}}{\text{Current Liabilities}}
$$

**Interpretation:** The most conservative liquidity measure — only cash is counted.

The cash ratio is most relevant in **distress scenarios** where receivables may be uncollectible and inventory unsaleable.  In normal operations, a cash ratio below 1.0 is perfectly acceptable because the firm expects to generate cash from operations and collections.

### 5.4 Defensive Interval Ratio

$$
\text{DIR} = \frac{\text{Cash} + \text{Receivables} + \text{Marketable Securities}}{\text{Daily Operating Expenditures}}
$$

where Daily Operating Expenditures = (COGS + SG&A - Depreciation) / 365.

**Interpretation:** The number of days a firm can operate using only its liquid assets, without any additional revenue.  A longer interval provides a greater cushion.

**Example calculation:** If a firm has \$150M in liquid assets and daily operating expenditures of \$3M, its defensive interval is 50 days — meaning it could survive nearly two months with zero revenue before exhausting its liquid assets.

> **Key Concept:** Liquidity ratios are *static* snapshots.  They do not capture the *timing* of cash inflows and outflows within the period.  Cash-flow analysis complements ratio analysis for a complete liquidity picture.

> **Common Mistake:** Forgetting to subtract depreciation from operating expenditures when calculating the defensive interval.  Depreciation is a non-cash expense — it does not require cash outflow, so it should not be included in daily operating expenditures.

#### Liquidity Ratio Summary

| Ratio | Formula | Best For |
|-------|---------|----------|
| Current Ratio | CA / CL | General screening |
| Quick Ratio | (Cash + AR) / CL | Firms with heavy inventory |
| Cash Ratio | Cash / CL | Distress analysis |
| Defensive Interval | Liquid Assets / Daily OpEx | Runway assessment |

In [ ]:
# ---------------------------------------------------------------
# 5. Liquidity Ratios
# ---------------------------------------------------------------

def liquidity_ratios(c):
    return {
        "Current Ratio":   c["current_assets"] / c["current_liab"],
        "Quick Ratio":     (c["cash"] + c["receivables"]) / c["current_liab"],
        "Cash Ratio":      c["cash"] / c["current_liab"],
        "Defensive Interval (days)": (c["cash"] + c["receivables"]) / c["daily_op_exp"],
    }

for c in companies:
    ratios = liquidity_ratios(c)
    print(f"\n{'='*62}")
    print(f"  {c['name']} — Liquidity Ratios")
    print(f"{'='*62}")
    print(f"{'Ratio':<30}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 62)
    for name, vals in ratios.items():
        print(f"{name:<30}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

#### Liquidity Ratios: Interpretation of Results

The liquidity charts above show how each company manages its short-term financial position.  When interpreting these results:

- **Current ratio above 2.0** generally indicates comfortable liquidity, but could also signal unproductive assets (excess cash, slow-moving inventory).
- **Quick ratio vs. current ratio gap:** A large gap between the current ratio and the quick ratio indicates heavy reliance on inventory.  For a manufacturing firm this is expected; for a services firm it may indicate inventory management problems.
- **Cash ratio trends:** A rising cash ratio over time might indicate the firm is hoarding cash (perhaps due to uncertain investment opportunities), while a declining cash ratio could signal tightening liquidity.

> **Key Concept:** The relationship between liquidity ratios tells a story.  If the current ratio is healthy but the cash ratio is dangerously low, the firm's liquidity depends heavily on its ability to collect receivables and liquidate inventory quickly — both of which may be impaired during a financial crisis.

> **CFA Exam Tip:** Remember that certain industries operate normally with current ratios below 1.0 (e.g., restaurants, fast food chains) because they collect cash from customers immediately while paying suppliers on credit terms.

#### What the Numbers Tell Us About Each Company

Systematically compare each company's liquidity profile:

1. **Most liquid company:** Which firm has the highest quick ratio?  This is the one best positioned to withstand a sudden revenue shortfall.
2. **Inventory-dependent company:** Which firm shows the largest gap between current and quick ratios?  This firm's liquidity is most vulnerable to inventory obsolescence.
3. **Trend direction:** Is liquidity improving or deteriorating?  A firm whose current ratio has declined from 2.5 to 1.5 over five years may be optimising working capital (positive) or gradually depleting its liquid buffer (negative) — context determines interpretation.
4. **Defensive interval:** A firm with 90+ days of defensive interval has substantial runway; a firm with only 20 days is operating with minimal buffer.

> **Common Mistake:** Treating all components of current assets as equally liquid.  Cash is immediately available, receivables take 30-90 days to collect, and inventory may take months to sell — especially during downturns when demand weakens.

#### The Liquidity Spectrum in Practice

In practice, analysts think of liquidity on a spectrum rather than as a binary pass/fail:

| Liquidity Level | Characteristics | Typical Industries |
|-----------------|----------------|--------------------|
| **Cash-rich** | Cash ratio > 1.0, minimal short-term debt | Technology, pharma with patent revenue |
| **Comfortable** | Current ratio > 2.0, quick ratio > 1.0 | Diversified industrials, consumer staples |
| **Adequate** | Current ratio 1.2-2.0, quick ratio 0.7-1.0 | Retail, light manufacturing |
| **Tight** | Current ratio < 1.2, quick ratio < 0.5 | Airlines, restaurants, highly seasonal businesses |
| **Distressed** | Current ratio < 0.8, declining cash | Firms approaching bankruptcy |

> **CFA Exam Tip:** A declining liquidity trend across multiple periods is more concerning than a single period of low liquidity.  The trend reveals whether the situation is temporary (seasonal dip) or structural (systematic deterioration).

In [ ]:
# ---------------------------------------------------------------
# Visualise: Liquidity ratios for latest year
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

labels = company_names
current = [c["current_assets"][-1] / c["current_liab"][-1] for c in companies]
quick   = [(c["cash"][-1] + c["receivables"][-1]) / c["current_liab"][-1] for c in companies]
cash_r  = [c["cash"][-1] / c["current_liab"][-1] for c in companies]

x = np.arange(len(labels))
w = 0.22
ax.bar(x - w, current, w, color=PRIMARY, label="Current Ratio")
ax.bar(x,     quick,   w, color=SECONDARY, label="Quick Ratio")
ax.bar(x + w, cash_r,  w, color=TERTIARY, label="Cash Ratio")

ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="1.0 threshold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Ratio (x)")
ax.set_title("Liquidity Ratios — Year 5 Snapshot")
ax.legend()
plt.tight_layout()
plt.show()

#### Additional Liquidity Analysis

The second set of liquidity charts provides the more conservative measures — quick ratio and cash ratio.  Comparing these with the current ratio reveals the composition of each firm's liquidity:

**Interpreting the liquidity hierarchy:**

| If Current Ratio Is... | And Quick Ratio Is... | Then... |
|------------------------|-----------------------|---------|
| High (>2.0) | Also high (>1.5) | Genuinely strong liquidity — ample liquid assets |
| High (>2.0) | Low (<0.8) | Liquidity depends on inventory — may be fragile in downturn |
| Moderate (1.0-1.5) | Close to current | Little inventory; liquidity comes from cash/receivables |
| Below 1.0 | Below 1.0 | Potential liquidity crisis — monitor closely, check credit lines |

> **Key Concept:** The progression from current ratio to quick ratio to cash ratio strips away increasingly illiquid assets, revealing the true "liquid core" of the firm's current asset base.  Each step provides a more conservative — and arguably more realistic — view of liquidity.

> **CFA Exam Tip:** When a firm's liquidity ratios are borderline, check its credit facilities.  Many firms maintain revolving credit lines (committed but undrawn) that provide liquidity not visible in balance-sheet ratios.  The notes to the financial statements disclose these facilities.

**The working capital management connection:**

Liquidity ratios are the output of working capital management decisions.  The key components are:

| Decision | Increases Liquidity | Decreases Liquidity |
|----------|--------------------|--------------------|
| Tighten credit terms | Faster collection → lower AR → higher quick ratio | May lose customers |
| Reduce inventory levels | Less cash tied up → higher ratios | Risk of stock-outs |
| Extend payable terms | Preserve cash → higher cash ratio | May damage supplier relationships |
| Draw on credit facility | More cash → higher cash ratio | Increases debt, future interest |

> **Common Mistake:** Viewing liquidity ratios in isolation from the rest of the financial profile.  A firm that improves its current ratio by aggressively cutting inventory may simultaneously damage its ability to fulfil customer orders.  Always check whether liquidity improvements come at the expense of operational capacity.

## 6. Solvency Ratios

Solvency ratios evaluate a firm's capacity to service its **long-term** debt obligations.

While liquidity focuses on the short term (can you pay this month's bills?), solvency focuses on the long term (can you service your debt for the next 10-20 years?).  Solvency ratios are critical for **fixed-income analysts** evaluating credit risk and for **equity analysts** assessing financial leverage.

The two main categories of solvency ratios are:

1. **Leverage ratios** — measure the *amount* of debt relative to equity or assets (D/E, D/A).  These tell you *how much* debt the firm has.
2. **Coverage ratios** — measure the firm's *ability to service* its debt obligations from operating cash flows (interest coverage, fixed charge coverage).  These tell you *whether* the firm can afford its debt.

> **Key Concept:** High leverage is a double-edged sword.  It magnifies returns in good times (because the firm earns on borrowed money) but amplifies losses in bad times (because interest payments are fixed).  The Modigliani-Miller theorem tells us that in a perfect market, capital structure is irrelevant — but taxes, bankruptcy costs, and agency problems make leverage very relevant in practice.

**The leverage-return amplification effect:**

Consider a firm with \$100M in assets earning 10% ROA:

| Scenario | Equity | Debt | Interest (5%) | Net Income | ROE |
|----------|--------|------|----------------|------------|-----|
| No leverage | \$100M | \$0 | \$0 | \$10M | 10.0% |
| Moderate leverage | \$50M | \$50M | \$2.5M | \$7.5M | 15.0% |
| High leverage | \$25M | \$75M | \$3.75M | \$6.25M | 25.0% |

The same \$100M in assets and 10% ROA produces ROEs ranging from 10% to 25% depending solely on capital structure.  But if ROA drops to 3%, the highly leveraged firm earns only \$0.75M on \$25M of equity (3% ROE minus interest), while the unlevered firm earns \$3M on \$100M (3% ROE).

| Ratio | Type | What It Reveals | Warning Threshold |
|-------|------|-----------------|-------------------|
| Debt-to-Equity | Leverage | Creditor vs. owner financing mix | > 2.0 for most industries |
| Debt-to-Assets | Leverage | Fraction of assets financed by debt | > 0.6 for most industries |
| Interest Coverage | Coverage | Operating income per dollar of interest | < 2.0 |
| Fixed Charge Coverage | Coverage | Operating income per dollar of all fixed charges | < 1.5 |

> **CFA Exam Tip:** Credit-rating agencies (S&P, Moody's, Fitch) rely heavily on solvency ratios.  An interest coverage ratio below 2.0 often triggers a downgrade watch.  Investment-grade bonds typically require interest coverage above 3.0.

### 6.1 Debt-to-Equity Ratio

$$
\text{D/E} = \frac{\text{Total Liabilities}}{\text{Shareholders' Equity}}
$$

**Interpretation:** The proportion of creditor financing relative to owner financing.  Higher D/E indicates greater financial leverage and, consequently, greater financial risk.

**D/E benchmarks by industry:**

| Industry | Typical D/E | Explanation |
|----------|-------------|-------------|
| Utilities | 1.0 - 2.5 | Capital-intensive, stable cash flows support debt |
| Banks | 5.0 - 15.0 | Highly leveraged by nature (deposits are liabilities) |
| Technology | 0.0 - 0.5 | Asset-light, strong cash generation |
| Real estate (REITs) | 0.5 - 1.5 | Property assets support moderate leverage |
| Airlines | 1.5 - 4.0 | Capital-intensive, volatile earnings |

> **Key Concept:** D/E must always be interpreted in the context of the industry.  A D/E of 2.0 would be alarming for a technology company but perfectly normal for a utility.  The key question is whether the firm's cash flows can comfortably service the debt, not whether the number itself is "high" or "low."

### 6.2 Debt-to-Assets Ratio

$$
\text{D/A} = \frac{\text{Total Liabilities}}{\text{Total Assets}}
$$

**Interpretation:** The fraction of assets financed by debt.  A D/A above 0.5 means creditors have provided more financing than equity holders.

**Mathematical relationship between D/E and D/A:**

$$
\text{D/A} = \frac{\text{D/E}}{1 + \text{D/E}}
$$

This means D/E of 1.0 corresponds to D/A of 0.50, D/E of 2.0 corresponds to D/A of 0.67, and D/E of 4.0 corresponds to D/A of 0.80.  The two ratios convey the same information but are scaled differently.

> **CFA Exam Tip:** The CFA curriculum uses both D/E and D/A, so be comfortable converting between them.  Remember: D/A can never exceed 1.0 (assets always exceed liabilities for a solvent firm), while D/E can be any positive number.

### 6.3 Interest Coverage Ratio

$$
\text{Interest Coverage} = \frac{\text{EBIT}}{\text{Interest Expense}}
$$

**Interpretation:** The number of times operating income covers interest payments.  A ratio below 1.5 is a strong warning signal; bond covenants often require coverage above 2.0 or 3.0.

**Interest coverage benchmarks by credit rating:**

| Credit Rating | Typical Interest Coverage | Interpretation |
|---------------|--------------------------|----------------|
| AAA | > 10.0x | Extremely strong coverage |
| AA | 6.0 - 10.0x | Very strong coverage |
| A | 4.0 - 6.0x | Strong coverage |
| BBB (investment grade floor) | 2.5 - 4.0x | Adequate coverage |
| BB (speculative) | 1.5 - 2.5x | Vulnerable |
| B or below | < 1.5x | High default risk |

### 6.4 Fixed Charge Coverage Ratio

$$
\text{Fixed Charge Coverage} = \frac{\text{EBIT} + \text{Lease Payments}}{\text{Interest Expense} + \text{Lease Payments}}
$$

**Interpretation:** Broadens interest coverage to include all fixed financial commitments (e.g., lease obligations).  This is especially important for firms with significant operating leases (airlines, retailers, restaurants).

> **CFA Exam Tip:** The CFA curriculum tests the fixed-charge coverage ratio specifically — do not skip it.  Under IFRS 16 and ASC 842, most leases are now capitalised on the balance sheet, but the fixed-charge coverage concept remains relevant for analytical purposes.

> **Key Concept:** A firm can have healthy interest coverage but poor fixed-charge coverage if it has large lease obligations.  Airlines, for example, may show 4x interest coverage but only 1.5x fixed-charge coverage once aircraft lease payments are included — a much less comfortable position.

In [ ]:
# ---------------------------------------------------------------
# 6. Solvency Ratios
# ---------------------------------------------------------------

def solvency_ratios(c):
    return {
        "Debt / Equity":        c["total_liab"] / c["equity"],
        "Debt / Assets":        c["total_liab"] / c["total_assets"],
        "Interest Coverage":    c["operating_income"] / c["interest_expense"],
        "Fixed Charge Coverage": (c["operating_income"] + c["lease_payments"]) /
                                 (c["interest_expense"] + c["lease_payments"]),
    }

for c in companies:
    ratios = solvency_ratios(c)
    print(f"\n{'='*62}")
    print(f"  {c['name']} — Solvency Ratios")
    print(f"{'='*62}")
    print(f"{'Ratio':<26}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 62)
    for name, vals in ratios.items():
        print(f"{name:<26}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

#### Solvency Ratios: Interpretation of Results

The solvency charts above reveal the capital-structure profiles of our three companies.  Key observations to look for:

- **D/E ratio trends:** A rising D/E ratio over time means the firm is becoming more leveraged.  This could be intentional (e.g., leveraged buyback, debt-funded acquisition) or a warning sign (e.g., accumulating losses eroding equity).
- **Interest coverage stability:** Volatile interest coverage is more concerning than a low but stable level.  A firm that consistently covers interest 3x is less risky than one that swings between 1.5x and 5x.
- **Peer comparison:** If one firm has dramatically different leverage than its peers, investigate why — it may reflect a different strategic choice, a different stage in the capital-allocation cycle, or constraints imposed by existing covenants.

> **Common Mistake:** Assuming lower leverage is always better.  Moderate leverage can enhance shareholder returns (ROE) without excessive risk.  The optimal leverage depends on the firm's business risk, cash-flow stability, and growth opportunities.

#### Solvency Diagnostic Framework

| Observation | Interpretation | Risk Level |
|-------------|----------------|------------|
| D/E rising + Coverage falling | Aggressive leveraging with declining profitability | High |
| D/E rising + Coverage stable | Strategic leverage increase, profitability absorbs interest | Moderate |
| D/E stable + Coverage rising | Improving profitability on stable capital base | Low |
| D/E falling + Coverage rising | Deleveraging with improving operations | Very Low |

> **CFA Exam Tip:** When analysing solvency, always examine both leverage ratios *and* coverage ratios together.  A firm with D/E of 3.0 but interest coverage of 8.0 is in a very different position from a firm with D/E of 3.0 but coverage of 1.5 — the first can comfortably service its debt, while the second is on the edge.

#### The Solvency-Profitability Nexus

Solvency and profitability are deeply interconnected through the DuPont framework.  The equity multiplier (Assets/Equity) is both a solvency indicator and a profitability amplifier.  This dual nature creates a fundamental tension in financial management:

- **More leverage** improves ROE (if ROA exceeds the after-tax cost of debt) but worsens solvency ratios.
- **Less leverage** improves solvency but reduces ROE for shareholders.

The optimal balance depends on the firm's business risk profile.  Firms with stable, predictable cash flows (utilities, consumer staples) can safely carry more leverage.  Firms with volatile, uncertain cash flows (technology, mining) should maintain lower leverage to preserve financial flexibility.

> **CFA Exam Tip:** When analysing a firm's capital structure, always consider the interaction between leverage and profitability.  The DuPont framework (Section 8) makes this interaction explicit and quantifiable.

In [ ]:
# ---------------------------------------------------------------
# Visualise: Debt-to-Equity trend
# ---------------------------------------------------------------
fig, ax = plt.subplots()
for c, col in zip(companies, colours):
    de = c["total_liab"] / c["equity"]
    ax.plot(years, de, marker="s", color=col, linewidth=2, label=c["name"])

ax.set_xlabel("Year")
ax.set_ylabel("Debt / Equity")
ax.set_title("Leverage (D/E) — 5-Year Trend")
ax.set_xticks(years)
ax.legend()
plt.tight_layout()
plt.show()

#### Additional Solvency Analysis

The second set of solvency charts shows coverage ratios — a critically important dimension that leverage ratios alone cannot capture.

**The distinction between leverage and coverage:**

- **Leverage ratios** (D/E, D/A) tell you *how much* debt the firm has relative to its equity or assets.
- **Coverage ratios** (interest coverage, fixed charge coverage) tell you whether the firm *can afford* to service that debt.

A firm can have high leverage but strong coverage if its operations generate substantial cash flow.  Conversely, a firm can have moderate leverage but weak coverage if its profitability is low.

> **Key Concept:** Coverage ratios are generally more informative than leverage ratios for assessing default risk, because they directly measure the firm's ability to meet its obligations.  A firm with D/E of 3.0 but interest coverage of 8.0x is in much better financial health than a firm with D/E of 1.5 but interest coverage of 1.2x.

**Warning signals in coverage ratios:**

| Interest Coverage | Signal | Action |
|-------------------|--------|--------|
| > 5.0x | Comfortable | Low default risk |
| 3.0 - 5.0x | Adequate | Monitor trends |
| 1.5 - 3.0x | Tight | Detailed analysis needed |
| < 1.5x | Danger zone | Assess refinancing options, asset sales |
| < 1.0x | Cannot cover interest | Distress imminent without corrective action |

> **CFA Exam Tip:** Bond covenants often specify minimum coverage ratios.  If a firm's interest coverage falls below the covenant threshold, it triggers a technical default — even if the firm is still making interest payments.  This can accelerate repayment and potentially force bankruptcy.

## 7. Valuation Ratios (Preview)

Valuation ratios relate a firm's **market price** to its fundamentals.  They are covered in depth under Equity Valuation; here we present a brief overview as a bridge from financial-statement analysis to security analysis.

Unlike the previous ratio families, valuation ratios incorporate **market data** — the stock price, market capitalisation, or enterprise value — alongside accounting data.  This makes them forward-looking in the sense that market prices embed investor expectations about future performance.

> **Key Concept:** Valuation ratios do not tell you whether a stock is "cheap" or "expensive" in absolute terms.  They provide *relative* valuation — a stock may trade at a low P/E because investors expect earnings to decline, not because it is undervalued.  Always pair valuation ratios with fundamental analysis to distinguish genuine bargains from value traps.

#### When to Use Each Multiple

| Multiple | Best Used When | Limitations |
|----------|---------------|-------------|
| P/E | Firm has positive, stable earnings | Meaningless when earnings are negative; distorted by one-time items |
| P/B | Asset-heavy firms (banks, real estate) | Less relevant for asset-light firms (tech, services) |
| P/S | Early-stage firms with negative earnings | Ignores profitability — a firm with high revenue but no profit still gets a low P/S |
| EV/EBITDA | Comparing firms with different leverage | Ignores capital expenditure needs; may overstate value for capex-heavy firms |

> **CFA Exam Tip:** The CFA curriculum distinguishes between *trailing* multiples (based on last 12 months' data) and *forward* multiples (based on analyst estimates for the next 12 months).  Forward multiples are generally considered more relevant for investment decisions because markets price in expectations, not history.

#### The Forward-Looking Nature of Valuation Ratios

Unlike profitability, activity, liquidity, and solvency ratios (which are all backward-looking), valuation ratios incorporate the market's forward-looking expectations.  A high P/E ratio does not necessarily mean a stock is overvalued — it may mean the market expects rapid earnings growth.  Similarly, a low P/E does not mean a stock is a bargain — it may reflect expectations of declining earnings.

This forward-looking element makes valuation ratios both more useful and more challenging to interpret:

- **More useful** because they embed the collective wisdom of all market participants about future prospects.
- **More challenging** because you must separate what the market expects from what you believe will actually happen.

> **Common Mistake:** Buying stocks solely because they have low P/E ratios.  Many low-P/E stocks are "value traps" — cheap for a good reason (declining industry, management problems, regulatory headwinds).  Always investigate *why* the market is assigning a low multiple before concluding that a stock is undervalued.

### 7.1 Price-to-Earnings (P/E)

$$
\text{P/E} = \frac{\text{Market Price per Share}}{\text{Earnings per Share}}
$$

**Interpretation:** How much investors are willing to pay per dollar of current earnings.  High P/E can indicate growth expectations or overvaluation.

**P/E ratio ranges and interpretation:**

| P/E Range | Typical Interpretation |
|-----------|----------------------|
| < 10 | Value stock or earnings expected to decline |
| 10 - 20 | Fairly valued for mature companies |
| 20 - 40 | Growth premium — investors expect earnings to grow significantly |
| > 40 | High-growth or speculative — earnings must grow dramatically to justify the price |
| Negative | Company has negative earnings (P/E is meaningless; use P/S or EV/EBITDA instead) |

> **Common Mistake:** Comparing P/E ratios across industries without adjusting for growth rates.  A technology company with a P/E of 35 growing earnings at 25% annually may be cheaper than a utility with a P/E of 15 growing at 3%.  The **PEG ratio** (P/E divided by earnings growth rate) addresses this by normalising for growth.

### 7.2 Price-to-Book (P/B)

$$
\text{P/B} = \frac{\text{Market Price per Share}}{\text{Book Value per Share}}
$$

**Interpretation:** Compares market valuation to accounting (book) value.  A P/B below 1.0 may signal undervaluation or fundamental weakness.

P/B is most useful for **asset-heavy industries** where book value is a meaningful proxy for intrinsic value (banks, insurance companies, real estate firms).  It is less useful for asset-light firms (technology, consulting) where the most valuable assets (intellectual property, human capital, brand) are not captured on the balance sheet.

### 7.3 Price-to-Sales (P/S)

$$
\text{P/S} = \frac{\text{Market Cap}}{\text{Revenue}}
$$

**Interpretation:** Useful for firms with negative earnings where P/E is undefined.  P/S is also less susceptible to accounting manipulation than P/E, since revenue is harder to manipulate than earnings.

### 7.4 EV/EBITDA

$$
\text{EV/EBITDA} = \frac{\text{Enterprise Value}}{\text{EBITDA}}
$$

where Enterprise Value = Market Cap + Total Debt - Cash.

> **Key Concept:** EV/EBITDA is capital-structure neutral (unlike P/E), making it a preferred multiple for comparing companies with different leverage levels.  Because enterprise value includes both equity and debt, and EBITDA is a pre-interest measure, the ratio is not distorted by financing decisions.

> **CFA Exam Tip:** The exam frequently asks when to use P/E versus EV/EBITDA.  Use P/E for equity-only comparisons within the same industry; use EV/EBITDA when comparing firms with different capital structures or evaluating acquisition targets (where the buyer acquires the whole enterprise, not just equity).

In [ ]:
# ---------------------------------------------------------------
# 7. Valuation Ratios
# ---------------------------------------------------------------

def valuation_ratios(c):
    eps = c["net_income"] / c["shares_outstanding"]
    bvps = c["equity"] / c["shares_outstanding"]
    ev = c["market_cap"] + c["long_term_debt"] - c["cash"]
    return {
        "EPS ($)":       eps,
        "P/E":           c["share_price"] / eps,
        "P/B":           c["share_price"] / bvps,
        "P/S":           c["market_cap"] / c["revenue"],
        "EV/EBITDA":     ev / c["ebitda"],
    }

for c in companies:
    ratios = valuation_ratios(c)
    print(f"\n{'='*55}")
    print(f"  {c['name']} — Valuation Ratios")
    print(f"{'='*55}")
    print(f"{'Ratio':<16}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 55)
    for name, vals in ratios.items():
        print(f"{name:<16}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

## 8. DuPont 3-Factor Decomposition

### 8.1 The Core Insight

Return on Equity is the product of three distinct drivers:

$$
\text{ROE} = \underbrace{\frac{\text{Net Income}}{\text{Revenue}}}_{\text{Net Profit Margin}} \;\times\; \underbrace{\frac{\text{Revenue}}{\text{Total Assets}}}_{\text{Asset Turnover}} \;\times\; \underbrace{\frac{\text{Total Assets}}{\text{Equity}}}_{\text{Equity Multiplier (Leverage)}}
$$

This decomposition was originally developed by **Donaldson Brown** at DuPont Corporation in the 1920s and remains one of the most powerful analytical tools in finance.

The genius of the DuPont framework is that it decomposes a single aggregate number (ROE) into three **economically distinct** components, each measuring a different dimension of performance:

1. **Net Profit Margin** captures the firm's ability to extract profit from revenue — a *profitability* metric.
2. **Asset Turnover** captures how intensively the firm uses its asset base — an *efficiency* metric.
3. **Equity Multiplier** captures the degree of financial leverage — a *risk* metric.

> **Key Concept:** Two firms can report identical ROEs but for entirely different reasons.  A luxury goods maker might achieve 20% ROE through high margins (15% net margin x 0.8 turnover x 1.67 multiplier), while a discount retailer achieves the same 20% ROE through high turnover (2% margin x 4.0 turnover x 2.5 multiplier).  The DuPont framework reveals these profoundly different business models.

> **Common Mistake:** Students sometimes treat the DuPont decomposition as an approximation.  It is not — it is an algebraic **identity** that holds exactly for every firm in every period.  The Revenue and Total Assets terms cancel perfectly, leaving NI/Equity = ROE.

#### Historical Context

The DuPont system was created in 1919 when F. Donaldson Brown, an electrical engineer turned financial analyst at DuPont, developed a formula to evaluate the company's diverse operations.  The framework was so successful that General Motors adopted it after DuPont acquired a controlling interest, and it eventually became the standard approach to ROE analysis across corporate finance, investment banking, and equity research.

#### The DuPont Identity as a Management Tool

Beyond analysis, the 3-factor decomposition serves as a **strategic planning framework**:

| Strategic Priority | DuPont Lever | Tactical Actions |
|-------------------|-------------|-----------------|
| Improve profitability | Net Profit Margin | Raise prices, reduce COGS, cut SG&A, optimise tax |
| Improve efficiency | Asset Turnover | Reduce working capital, sell idle assets, improve capacity utilisation |
| Optimise capital structure | Equity Multiplier | Increase debt (if ROA > cost of debt), share buybacks |

> **CFA Exam Tip:** The DuPont decomposition is one of the most frequently tested topics in CFA Level 1 Financial Statement Analysis.  Be prepared to compute all three factors, verify they multiply to ROE, and interpret each factor in the context of a specific company or industry.

### 8.2 Algebraic Derivation

Start from the definition of ROE:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Equity}}
$$

**Step 1:** Multiply and divide by Revenue:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Equity}} \times \frac{\text{Revenue}}{\text{Revenue}}
$$

**Step 2:** Multiply and divide by Total Assets:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Equity}} \times \frac{\text{Revenue}}{\text{Revenue}} \times \frac{\text{Total Assets}}{\text{Total Assets}}
$$

**Step 3:** Rearrange — group numerators and denominators strategically:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} \times \frac{\text{Total Assets}}{\text{Equity}}
$$

**Step 4:** Verify the cancellation chain:

$$
\frac{\text{NI}}{\cancel{\text{Rev}}} \times \frac{\cancel{\text{Rev}}}{\cancel{\text{Assets}}} \times \frac{\cancel{\text{Assets}}}{\text{Equity}} = \frac{\text{NI}}{\text{Equity}} = \text{ROE} \quad \checkmark
$$

> **Key Concept:** When Revenue appears in both the numerator of Asset Turnover and the denominator of Net Profit Margin, the two Revenue terms cancel.  Similarly, Total Assets cancels between Asset Turnover and the Equity Multiplier.  The result is simply NI/Equity — the definition of ROE.

#### Worked Numerical Example

Consider a firm with the following data:

| Item | Value |
|------|-------|
| Net Income | \$120M |
| Revenue | \$1,000M |
| Total Assets | \$800M |
| Shareholders' Equity | \$400M |

**Direct ROE calculation:**

$$
\text{ROE} = \frac{120}{400} = 30\%
$$

**DuPont decomposition:**

$$
\text{Net Margin} = \frac{120}{1000} = 12\%
$$

$$
\text{Asset Turnover} = \frac{1000}{800} = 1.25\times
$$

$$
\text{Equity Multiplier} = \frac{800}{400} = 2.0\times
$$

$$
\text{ROE} = 12\% \times 1.25 \times 2.0 = 30\% \quad \checkmark
$$

**Diagnosis:** This firm earns a strong 30% ROE.  Roughly half comes from leverage (multiplier of 2.0) and the remainder from solid operating performance (12% margins with reasonable turnover).  If the equity multiplier were reduced to 1.5 (less debt), ROE would fall to 22.5% — still healthy, and with lower financial risk.

### 8.3 Interpreting the Three Factors

| Factor | Measures | Improved by | Typical Range |
|--------|----------|-------------|---------------|
| **Net Profit Margin** | Profitability | Raising prices, cutting costs, tax efficiency | 2% (retail) to 30% (software) |
| **Asset Turnover** | Efficiency | Generating more revenue per dollar of assets | 0.3x (utilities) to 3.0x (retail) |
| **Equity Multiplier** | Leverage | Using more debt (increases risk) | 1.0x (no debt) to 5.0x+ (banks) |

> **CFA Exam Tip:** A rising ROE driven primarily by an increasing equity multiplier signals growing financial risk, not improving operations.  Always decompose ROE before drawing conclusions about company performance.

The beauty of the DuPont framework is that it converts a single opaque number (ROE) into an *actionable diagnostic*.  Management teams and analysts can identify which lever to pull:

- **Low margin?** Focus on cost reduction or pricing strategy.
- **Low turnover?** Improve asset utilisation — reduce excess inventory, collect receivables faster.
- **High multiplier?** The ROE may be artificially inflated by leverage; check solvency ratios.

#### The Margin-Turnover Trade-off

One of the most important insights from DuPont analysis is the **inverse relationship** between margins and turnover across industries:

| Business Model | Net Margin | Asset Turnover | Product |
|---------------|------------|----------------|---------|
| Luxury goods (Hermes) | Very high | Very low | Margin x Turnover ≈ ROA |
| Mass retail (Walmart) | Very low | Very high | Margin x Turnover ≈ ROA |
| Balanced (industrial) | Moderate | Moderate | Margin x Turnover ≈ ROA |

Different companies can achieve similar ROA (and ROE) through very different combinations.  There is no single "correct" mix — it depends on the firm's competitive strategy, industry dynamics, and customer base.

> **Key Concept:** The DuPont framework is the most elegant tool in financial analysis because it simultaneously answers three questions: How profitable is the firm? How efficient is the firm? How leveraged is the firm?  No other single framework provides this breadth of insight from a single equation.

### 9.2 Algebraic Derivation

Begin with the 3-factor identity:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} \times \frac{\text{Total Assets}}{\text{Equity}}
$$

Now decompose Net Profit Margin by multiplying and dividing by EBT and EBIT:

$$
\frac{\text{Net Income}}{\text{Revenue}} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{EBT}}{\text{EBT}} \times \frac{\text{EBIT}}{\text{EBIT}}
$$

Rearrange the right side:

$$
\frac{\text{Net Income}}{\text{Revenue}} = \frac{\text{Net Income}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Revenue}}
$$

Substituting back into the 3-factor identity yields the full 5-factor decomposition:

$$
\text{ROE} = \frac{\text{NI}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Rev}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

**Cancellation verification:**

$$
\frac{\text{NI}}{\cancel{\text{EBT}}} \times \frac{\cancel{\text{EBT}}}{\cancel{\text{EBIT}}} \times \frac{\cancel{\text{EBIT}}}{\cancel{\text{Rev}}} \times \frac{\cancel{\text{Rev}}}{\cancel{\text{Assets}}} \times \frac{\cancel{\text{Assets}}}{\text{Equity}} = \frac{\text{NI}}{\text{Equity}} = \text{ROE} \quad \checkmark
$$

#### Understanding Each Component in Depth

**Tax Burden (NI/EBT):** This ratio equals $(1 - \text{effective tax rate})$.  If the effective tax rate is 25%, the tax burden is 0.75, meaning 75% of pre-tax profit survives taxation.  A higher value is better.

Factors that affect the tax burden ratio:
- **Statutory corporate tax rate** in the firm's home country.
- **Tax credits** (R&D credits, investment tax credits).
- **Tax-loss carryforwards** from prior years.
- **Geographic income mix** — profits earned in low-tax jurisdictions improve the ratio.
- **Permanent differences** (e.g., non-deductible fines, tax-exempt income).

**Interest Burden (EBT/EBIT):** This ratio measures the proportion of operating income remaining after interest payments.  For an all-equity firm with no debt, this ratio equals 1.0.  As debt increases, interest expense rises, and this ratio falls toward zero.

The interest burden can be expressed as:

$$
\frac{\text{EBT}}{\text{EBIT}} = \frac{\text{EBIT} - \text{Interest Expense}}{\text{EBIT}} = 1 - \frac{\text{Interest Expense}}{\text{EBIT}}
$$

This shows that the interest burden is the complement of the interest-expense-to-EBIT ratio.

**Operating Margin (EBIT/Revenue):** The purest measure of operational efficiency — it excludes both financing and tax effects.  This is the factor that best reflects management's operational performance.

> **Common Mistake:** Students sometimes assume the Interest Burden ratio and the Interest Coverage ratio measure the same thing.  They do not.  Interest Coverage = EBIT/Interest (a *times* measure, typically 2-10x), while Interest Burden = EBT/EBIT = (EBIT - Interest)/EBIT (a *proportion* measure between 0 and 1).

#### Worked Numerical Example

| Item | Value |
|------|-------|
| Revenue | \$2,000M |
| EBIT | \$300M |
| Interest Expense | \$50M |
| EBT | \$250M (= EBIT - Interest) |
| Taxes | \$62.5M (25% rate) |
| Net Income | \$187.5M |
| Total Assets | \$1,500M |
| Equity | \$600M |

**5-factor calculation:**

| Factor | Calculation | Value | Interpretation |
|--------|-------------|-------|----------------|
| Tax Burden | 187.5 / 250 | 0.750 | 25% effective tax rate |
| Interest Burden | 250 / 300 | 0.833 | 16.7% of EBIT goes to interest |
| Operating Margin | 300 / 2000 | 0.150 | 15 cents of every revenue dollar is EBIT |
| Asset Turnover | 2000 / 1500 | 1.333 | Each dollar of assets generates \$1.33 of revenue |
| Equity Multiplier | 1500 / 600 | 2.500 | Total assets are 2.5x equity |

$$
\text{ROE} = 0.750 \times 0.833 \times 0.150 \times 1.333 \times 2.500 = 31.25\%
$$

**Verify directly:** $187.5 / 600 = 31.25\%$ \checkmark

#### Sensitivity Analysis: What If Scenarios

Starting from the example above, consider how ROE changes if each factor shifts:

| Scenario | Changed Factor | New Value | New ROE | Change |
|----------|---------------|-----------|---------|--------|
| Base case | — | — | 31.25% | — |
| Tax cut (to 20%) | Tax Burden: 0.800 | 0.800 | 33.33% | +2.08% |
| Rate hike (interest +50bp) | Int Burden: 0.800 | 0.800 | 30.00% | -1.25% |
| Margin compression | Op Margin: 0.130 | 0.130 | 27.08% | -4.17% |
| New factory comes online | Turnover: 1.200 | 1.200 | 28.13% | -3.12% |
| Share buyback | Eq Mult: 3.000 | 3.000 | 37.50% | +6.25% |

> **Key Concept:** This sensitivity analysis reveals that the equity multiplier has the largest absolute impact on ROE — but it also carries the most risk.  The operating margin change has the second-largest impact and is the most directly controllable factor.

### DuPont 3-Factor: Key Observations

The tree diagram above makes the multiplicative structure of ROE concrete.  Notice how the same line items (Revenue, Total Assets) appear in *multiple* branches — this is not redundancy but reflects the algebraic cancellation that makes the identity work.

> **Key Concept:** When Revenue appears in both the numerator of Asset Turnover and the denominator of Net Profit Margin, the two Revenue terms cancel, leaving NI/Assets.  Multiply by Assets/Equity and you recover NI/Equity = ROE.  The decomposition is an **identity**, not an approximation — it holds exactly for every firm in every period.

#### Reading the Results

When interpreting the DuPont bar charts or factor tables:

- **Compare across companies:** Which firm has the highest margin?  The highest turnover?  The highest leverage?  The firm with the highest ROE may not lead in any single category but has the best *combination*.
- **Compare across time:** Are factors trending in favourable or unfavourable directions?  A gradually rising equity multiplier with stable margins suggests management is deliberately increasing leverage — understand whether this is strategic (funded growth) or distressed (covering operating shortfalls).
- **Look for trade-offs:** The margin-turnover trade-off is especially important.  A strategy shift from premium pricing to volume pricing shows up as declining margins but rising turnover.

#### Year-over-Year Change Analysis

To analyse how ROE changed from one year to the next, compute the contribution of each factor:

1. Calculate each DuPont factor for both years.
2. Compute the percentage change in each factor.
3. The factor with the largest percentage change is the primary driver of the ROE change.
4. Verify by checking whether the direction of the factor change is consistent with the direction of the ROE change.

> **CFA Exam Tip:** On the exam, when asked to "explain the change in ROE," always start with the DuPont decomposition.  Identify which factor changed most and provide an economic rationale for that change (e.g., "the decline in ROE was primarily driven by a fall in operating margins due to rising input costs").

In [ ]:
# ---------------------------------------------------------------
# Visualise: DuPont tree for AlphaCorp Year 5
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("DuPont 3-Factor Tree — AlphaCorp, Year 5", fontsize=14, fontweight="bold")

npm_a, ato_a, em_a, roe_a = dupont_3(alpha)

# Positions: (x, y)
boxes = {
    "ROE":    (5, 8.5, f"ROE\n{roe_a[-1]:.2%}"),
    "NPM":    (2, 6, f"Net Profit Margin\n{npm_a[-1]:.2%}"),
    "ATO":    (5, 6, f"Asset Turnover\n{ato_a[-1]:.3f}x"),
    "EM":     (8, 6, f"Equity Multiplier\n{em_a[-1]:.3f}x"),
    "NI":     (1, 3.5, f"Net Income\n${alpha['net_income'][-1]:,.0f}M"),
    "Rev1":   (3, 3.5, f"Revenue\n${alpha['revenue'][-1]:,.0f}M"),
    "Rev2":   (4, 3.5, f"Revenue\n${alpha['revenue'][-1]:,.0f}M"),
    "TA1":    (6, 3.5, f"Total Assets\n${alpha['total_assets'][-1]:,.0f}M"),
    "TA2":    (7, 3.5, f"Total Assets\n${alpha['total_assets'][-1]:,.0f}M"),
    "EQ":     (9, 3.5, f"Equity\n${alpha['equity'][-1]:,.0f}M"),
}

box_style = dict(boxstyle="round,pad=0.4", facecolor="lightyellow", edgecolor="grey")
for key, (bx, by, txt) in boxes.items():
    ax.text(bx, by, txt, ha="center", va="center", fontsize=9, bbox=box_style)

# Connectors
connectors = [
    ("ROE", "NPM"), ("ROE", "ATO"), ("ROE", "EM"),
    ("NPM", "NI"), ("NPM", "Rev1"),
    ("ATO", "Rev2"), ("ATO", "TA1"),
    ("EM", "TA2"), ("EM", "EQ"),
]
for parent, child in connectors:
    px, py, _ = boxes[parent]
    cx, cy, _ = boxes[child]
    ax.annotate("", xy=(cx, cy + 0.6), xytext=(px, py - 0.6),
                arrowprops=dict(arrowstyle="->", color="grey", lw=1.2))

# Multiplication signs
ax.text(3.5, 6, r"$\times$", ha="center", va="center", fontsize=16, color="red")
ax.text(6.5, 6, r"$\times$", ha="center", va="center", fontsize=16, color="red")

plt.tight_layout()
plt.show()

## 9. DuPont 5-Factor Decomposition

### 9.1 Extending the Framework

The 3-factor model lumps together tax effects, interest burden, and operating profitability into a single "net profit margin" term.  The **5-factor DuPont** model unpacks net profit margin into three sub-components:

$$
\text{ROE} = \underbrace{\frac{\text{Net Income}}{\text{EBT}}}_{\text{Tax Burden}} \;\times\; \underbrace{\frac{\text{EBT}}{\text{EBIT}}}_{\text{Interest Burden}} \;\times\; \underbrace{\frac{\text{EBIT}}{\text{Revenue}}}_{\text{Operating Margin}} \;\times\; \underbrace{\frac{\text{Revenue}}{\text{Total Assets}}}_{\text{Asset Turnover}} \;\times\; \underbrace{\frac{\text{Total Assets}}{\text{Equity}}}_{\text{Equity Multiplier}}
$$

#### Why Five Factors?

The 3-factor model's Net Profit Margin conflates three distinct effects:

1. **How well the firm operates** (operating efficiency).
2. **How much of operating profit goes to creditors** (financing decisions).
3. **How much of pre-tax profit goes to the government** (tax policy/jurisdiction).

By splitting Net Margin into Tax Burden × Interest Burden × Operating Margin, the 5-factor model allows analysts to disentangle these effects and provide much more targeted diagnoses.

> **Key Concept:** The 5-factor decomposition is especially valuable for cross-border comparisons.  A French company paying 33% corporate tax and a Singaporean company paying 17% corporate tax will have different tax burden ratios even if their operations are identical.  The 5-factor model isolates this effect so it does not contaminate the operational comparison.

#### The Progression from 3-Factor to 5-Factor

The 5-factor model is a strict refinement of the 3-factor model — they are fully compatible:

$$
\underbrace{\frac{\text{NI}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Rev}}}_{\text{Net Profit Margin (3-factor)}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

The first three terms of the 5-factor model multiply together to give the Net Profit Margin of the 3-factor model.  This means you can always "collapse" the 5-factor back to the 3-factor by multiplying the first three terms.  Use the 3-factor for quick diagnostics and the 5-factor when you need deeper insight into *why* the net margin is what it is.

> **CFA Exam Tip:** Be comfortable moving between the 3-factor and 5-factor decompositions.  The exam may give you 5-factor data and ask for the 3-factor decomposition, or vice versa.  The key relationship is: Net Margin = Tax Burden x Interest Burden x Operating Margin.

### 9.2 Algebraic Derivation

Begin with the 3-factor identity:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} \times \frac{\text{Total Assets}}{\text{Equity}}
$$

Now decompose Net Profit Margin by multiplying and dividing by EBT and EBIT:

$$
\frac{\text{Net Income}}{\text{Revenue}} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{EBT}}{\text{EBT}} \times \frac{\text{EBIT}}{\text{EBIT}}
$$

Rearrange the right side:

$$
\frac{\text{Net Income}}{\text{Revenue}} = \frac{\text{Net Income}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Revenue}}
$$

Substituting back into the 3-factor identity yields the full 5-factor decomposition:

$$
\text{ROE} = \frac{\text{NI}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Rev}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

**Cancellation verification:**

$$
\frac{\text{NI}}{\cancel{\text{EBT}}} \times \frac{\cancel{\text{EBT}}}{\cancel{\text{EBIT}}} \times \frac{\cancel{\text{EBIT}}}{\cancel{\text{Rev}}} \times \frac{\cancel{\text{Rev}}}{\cancel{\text{Assets}}} \times \frac{\cancel{\text{Assets}}}{\text{Equity}} = \frac{\text{NI}}{\text{Equity}} = \text{ROE} \quad \checkmark
$$

#### Understanding Each Component

**Tax Burden (NI/EBT):** This ratio equals $(1 - \text{effective tax rate})$.  If the effective tax rate is 25%, the tax burden is 0.75, meaning 75% of pre-tax profit survives taxation.  A higher value is better.

**Interest Burden (EBT/EBIT):** This ratio measures the proportion of operating income remaining after interest payments.  For an all-equity firm with no debt, this ratio equals 1.0.  As debt increases, interest expense rises, and this ratio falls toward zero.

**Operating Margin (EBIT/Revenue):** The purest measure of operational efficiency — it excludes both financing and tax effects.

> **Common Mistake:** Students sometimes assume the Interest Burden ratio and the Interest Coverage ratio measure the same thing.  They do not.  Interest Coverage = EBIT/Interest (a *times* measure), while Interest Burden = EBT/EBIT = (EBIT - Interest)/EBIT (a *proportion* measure between 0 and 1).

#### Worked Numerical Example

| Item | Value |
|------|-------|
| Revenue | \$2,000M |
| EBIT | \$300M |
| Interest Expense | \$50M |
| EBT | \$250M (= EBIT - Interest) |
| Taxes | \$62.5M (25% rate) |
| Net Income | \$187.5M |
| Total Assets | \$1,500M |
| Equity | \$600M |

**5-factor calculation:**

| Factor | Calculation | Value |
|--------|-------------|-------|
| Tax Burden | 187.5 / 250 | 0.750 |
| Interest Burden | 250 / 300 | 0.833 |
| Operating Margin | 300 / 2000 | 0.150 |
| Asset Turnover | 2000 / 1500 | 1.333 |
| Equity Multiplier | 1500 / 600 | 2.500 |

$$
\text{ROE} = 0.750 \times 0.833 \times 0.150 \times 1.333 \times 2.500 = 31.25\%
$$

**Verify directly:** $187.5 / 600 = 31.25\%$ \checkmark

### 9.3 Interpreting the Five Factors

| Factor | Formula | Measures | Range | Direction |
|--------|---------|----------|-------|-----------|
| **Tax Burden** | NI / EBT | How much EBT survives taxation | 0 to 1 (higher = lower tax rate) | Higher is better |
| **Interest Burden** | EBT / EBIT | How much EBIT survives interest costs | 0 to 1 (higher = lower interest cost) | Higher is better |
| **Operating Margin** | EBIT / Revenue | Core operational profitability | varies by industry | Higher is better |
| **Asset Turnover** | Revenue / Assets | Capital efficiency | varies by industry | Higher is better (generally) |
| **Equity Multiplier** | Assets / Equity | Financial leverage | >= 1 (higher = more debt) | Ambiguous — depends on context |

> **Key Concept:** The 5-factor model lets you pinpoint *exactly* why ROE changed.  Did the tax rate go up?  Did interest expense grow?  Did operating efficiency improve?  Each question maps to one factor.

> **Common Mistake:** Confusing the "interest burden" direction — a *lower* interest burden ratio means *more* of EBIT is consumed by interest, which is *worse*, not better.

### 9.4 Practical Use Cases

The 5-factor DuPont is especially useful for:

- **Year-over-year analysis:** If ROE dropped, which factor drove the decline?
- **Cross-border comparison:** Companies in different tax jurisdictions will differ in the tax-burden factor.
- **Leverage analysis:** Separating the interest burden from the equity multiplier reveals the *cost* of leverage, not just its presence.

#### Example: Diagnosing an ROE Decline

Suppose a firm's ROE fell from 18% to 14% year-over-year.  The 5-factor analysis reveals:

| Factor | Year 1 | Year 2 | Change |
|--------|--------|--------|--------|
| Tax Burden | 0.75 | 0.70 | -6.7% (tax rate increased) |
| Interest Burden | 0.85 | 0.80 | -5.9% (interest costs rose) |
| Operating Margin | 12.0% | 11.5% | -4.2% (slight margin compression) |
| Asset Turnover | 1.40 | 1.35 | -3.6% (slightly less efficient) |
| Equity Multiplier | 2.00 | 2.00 | 0.0% (leverage unchanged) |

**Diagnosis:** The ROE decline is broad-based, with the largest individual contributors being the increased tax burden (possibly due to a tax law change or shift in geographic income mix) and rising interest costs (possibly due to recent debt issuance or higher floating rates).  Operating performance also softened slightly.  The good news: leverage did not increase, so the firm is not trying to mask operational weakness with financial engineering.

> **CFA Exam Tip:** The CFA Level 1 curriculum explicitly tests the 5-factor decomposition.  Practice computing all five factors from a given set of financial statements.  A typical exam question gives you two years of data and asks you to identify the primary driver of the ROE change.

In [ ]:
# ---------------------------------------------------------------
# 9. DuPont 5-Factor Decomposition
# ---------------------------------------------------------------

def dupont_5(c):
    tax_burden      = c["net_income"] / c["ebt"]                # NI / EBT
    interest_burden = c["ebt"] / c["operating_income"]           # EBT / EBIT
    operating_margin = c["operating_income"] / c["revenue"]      # EBIT / Revenue
    asset_turnover  = c["revenue"] / c["total_assets"]           # Revenue / Assets
    equity_mult     = c["total_assets"] / c["equity"]            # Assets / Equity
    roe = tax_burden * interest_burden * operating_margin * asset_turnover * equity_mult
    return {
        "Tax Burden":       tax_burden,
        "Interest Burden":  interest_burden,
        "Operating Margin": operating_margin,
        "Asset Turnover":   asset_turnover,
        "Equity Multiplier": equity_mult,
        "ROE (5-factor)":   roe,
    }

for c in companies:
    factors = dupont_5(c)
    roe_direct = c["net_income"] / c["equity"]
    print(f"\n{'='*70}")
    print(f"  {c['name']} — 5-Factor DuPont Decomposition")
    print(f"{'='*70}")
    print(f"{'Factor':<22}", end="")
    for y in years:
        print(f"    Yr{y:d}", end="")
    print()
    print("-" * 70)
    for name, vals in factors.items():
        print(f"{name:<22}", end="")
        for v in vals:
            print(f" {v:7.4f}", end="")
        print()
    # Verification
    print(f"{'ROE (direct)':<22}", end="")
    for v in roe_direct:
        print(f" {v:7.4f}", end="")
    print()
    match_all = np.allclose(factors["ROE (5-factor)"], roe_direct, atol=ATOL)
    print(f"  Verification: {'PASS' if match_all else 'FAIL'}")

#### DuPont 5-Factor: Observations from the Visualisation

The 5-factor decomposition visualisation above highlights the granularity gained by splitting net margin into its constituent parts.  Compare the two DuPont views:

- The **3-factor view** tells you "the margin is low" but does not explain why.
- The **5-factor view** tells you *whether* the low margin is due to high taxes, heavy interest expense, or weak operating performance — each requiring a completely different remedial action.

For instance, if a firm's ROE declined year-over-year:

| Scenario | Factor Change | Diagnosis | Recommended Action |
|----------|---------------|-----------|-------------------|
| Tax reform | Tax burden fell | Government policy, not operations | Adjust forecasts; no operational change needed |
| Debt refinancing at higher rates | Interest burden fell | Higher interest costs | Evaluate refinancing or deleveraging options |
| Cost overruns | Operating margin fell | Core business deterioration | Investigate cost structure and pricing |
| Capacity expansion | Asset turnover fell | Assets grew faster than revenue | Allow time for new assets to ramp up |
| Share buyback | Equity multiplier rose | Financial engineering | Assess sustainability of leverage |

> **CFA Exam Tip:** When the exam asks "explain the change in ROE using the 5-factor DuPont model," compute each factor for both periods, identify which factor(s) changed most, and provide an economic interpretation for each change.  Simply listing the numbers without interpretation will not earn full marks.

#### Connecting the 5-Factor Model to Strategy

Each DuPont factor maps to a different area of management responsibility:

| Factor | Responsible Function | Strategic Levers |
|--------|---------------------|-----------------|
| Tax Burden | CFO / Tax team | Tax planning, jurisdiction choice, R&D credits |
| Interest Burden | CFO / Treasury | Debt structure, refinancing, hedging |
| Operating Margin | COO / Business units | Pricing, cost control, product mix, efficiency |
| Asset Turnover | COO / Operations | Working capital management, capex discipline |
| Equity Multiplier | CFO / Board | Capital structure policy, buybacks, dividends |

> **Key Concept:** The 5-factor DuPont decomposition is not just an analytical tool — it is a management accountability framework.  Each factor can be assigned to a specific executive or function, making it clear who is responsible for driving (or dragging) ROE.

In [ ]:
# ---------------------------------------------------------------
# Tornado chart: sensitivity of ROE to each DuPont factor (Year 5)
# ---------------------------------------------------------------

def tornado_dupont(c, year_idx=-1):
    """
    For each DuPont factor, compute how much ROE would change
    if that factor increased by 10%, holding all others constant.
    """
    factors_dict = dupont_5(c)
    factor_names = ["Tax Burden", "Interest Burden", "Operating Margin",
                    "Asset Turnover", "Equity Multiplier"]
    base_roe = factors_dict["ROE (5-factor)"][year_idx]
    base_vals = [factors_dict[f][year_idx] for f in factor_names]

    deltas = []
    for i in range(5):
        shocked = base_vals.copy()
        shocked[i] *= 1.10  # +10%
        new_roe = np.prod(shocked)
        deltas.append(new_roe - base_roe)

    return factor_names, deltas, base_roe


fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, c, col in zip(axes, companies, colours):
    names, deltas, base = tornado_dupont(c)
    # Sort by absolute impact
    order = np.argsort(np.abs(deltas))
    sorted_names = [names[i] for i in order]
    sorted_deltas = [deltas[i] for i in order]

    bars = ax.barh(sorted_names, [d * 100 for d in sorted_deltas], color=col, edgecolor="white")
    ax.set_xlabel("Change in ROE (pp)")
    ax.set_title(f"{c['name']}\nBase ROE = {base:.2%}")
    ax.axvline(0, color="black", linewidth=0.8)

fig.suptitle("Tornado Chart — ROE Sensitivity to +10% Shock in Each DuPont Factor (Year 5)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 10. Cross-Sectional vs. Time-Series Analysis

### 10.1 Two Dimensions of Comparison

Ratio analysis is most powerful when applied along **two axes**:

1. **Cross-sectional (peer comparison):** Comparing the same ratio across multiple companies *at a single point in time*.  This answers: "Is this company better or worse than its peers?"
2. **Time-series (trend analysis):** Tracking one company's ratio over multiple periods.  This answers: "Is this company improving or deteriorating?"

The most insightful analysis uses **both** dimensions simultaneously — a company might have the best ROE among its peers (strong cross-sectionally) but show a declining trend (weakening over time).  This combination of signals is far more informative than either dimension alone.

> **Key Concept:** A ratio in isolation is almost meaningless.  The number 15% for ROE tells you nothing without context.  Is 15% good?  Compared to what?  Industry peers averaging 10% — then yes.  Compared to the same company's 22% ROE three years ago — then no, something is declining.

#### Combining Both Dimensions

| Analysis Type | Question | Benchmark | Strength | Weakness |
|---------------|----------|-----------|----------|----------|
| Cross-sectional only | "How does this firm compare to peers today?" | Industry median, quartile rankings | Identifies relative positioning | Ignores trajectory |
| Time-series only | "Is this firm improving over time?" | Its own historical values | Shows momentum and trends | Ignores competitive context |
| Combined | "Is this firm's competitive position strengthening?" | Both peer medians and own history | Most complete picture | Requires more data and analysis |

#### The Four Quadrants of Ratio Analysis

When combining cross-sectional ranking with time-series trend, four scenarios emerge:

| | Above Peer Median | Below Peer Median |
|---|---|---|
| **Improving Trend** | Star performer: leading peers and improving | Recovering: below peers but catching up |
| **Deteriorating Trend** | Fading leader: still above peers but declining | Laggard: below peers and getting worse |

> **CFA Exam Tip:** The CFA curriculum specifically tests candidates on the ability to distinguish between cross-sectional and time-series analysis and to describe the limitations and appropriate applications of each approach.  The "four quadrants" framework is a useful mental model for structuring your analysis.

### 10.2 Methodology

- **Select comparable companies** — same industry, similar size, same accounting standards.
- **Normalise for accounting differences** — adjust for FIFO/LIFO, lease capitalisation, one-time items.
- **Use median (not mean) for peer benchmarks** — medians are robust to outliers.
- **Consistent time periods** — ensure all companies report for the same fiscal period.
- **Minimum peer group size** — aim for at least 5-10 companies for meaningful cross-sectional analysis.

### 10.3 Pitfalls

> **Common Mistake:** Comparing a software company's asset turnover with that of a utility company.  Industry composition dominates most ratios, so cross-sectional analysis must be done *within* a well-defined peer group.

Additional pitfalls:

- **Survivorship bias** — if you only look at companies that survived to Year 5, the sample is upward-biased.
- **Changing accounting standards** — IFRS 16 (lease capitalisation) significantly altered leverage and asset ratios.
- **Mergers and divestitures** — can cause discontinuities in time-series data.
- **Size effects** — larger firms may have structural advantages (economies of scale) that bias comparisons.
- **Geographic mix** — firms operating in different regions face different input costs, tax rates, and regulatory environments.

> **CFA Exam Tip:** The curriculum specifically asks candidates to distinguish between cross-sectional and time-series analysis and describe the limitations of each.  Be prepared to name at least three limitations for each approach.

#### Best Practices for Robust Ratio Analysis

1. **Always state your assumptions** — which time period, which peer group, which accounting adjustments.
2. **Present ranges, not just point estimates** — show the interquartile range (25th to 75th percentile) for peer benchmarks, not just the median.
3. **Investigate outliers** — before removing outliers from the peer group, understand *why* they are outliers.  Sometimes the outlier is the most interesting data point.
4. **Update regularly** — financial conditions change.  A ratio analysis from 12 months ago may no longer be relevant.
5. **Triangulate** — use multiple ratios from different families to confirm your conclusions.  If profitability ratios say the firm is strong but liquidity ratios flash warnings, investigate further.

### 10.4 Radar (Spider) Charts

A radar chart is a popular way to visualise multiple ratios simultaneously for several companies.  Each spoke represents a ratio, and each company is a polygon.  The further a vertex is from the centre, the "better" (or higher) that ratio is.

> **Key Concept:** Radar charts are useful for *qualitative* comparison — they give an at-a-glance profile — but can be misleading if the scales are not normalised.  We normalise each ratio to the [0, 1] range across the companies being compared.

#### How to Read a Radar Chart

1. **Shape symmetry:** A symmetric polygon indicates balanced performance across all dimensions.
2. **Shape size:** A larger overall polygon indicates stronger overall performance.
3. **Pointed vertices:** A polygon that spikes outward on one or two spokes but collapses inward on others reveals a company that excels in some areas but is weak in others.
4. **Overlapping regions:** Where two company polygons overlap, those firms have similar performance on those ratios.

#### Limitations of Radar Charts

- **Spoke ordering matters:** Rearranging the spokes changes the visual appearance of the polygon, potentially creating different impressions from the same data.
- **Area is misleading:** The visual area of the polygon depends on spoke ordering and does not have a meaningful quantitative interpretation.
- **Too many spokes:** With more than 8-10 spokes, the chart becomes cluttered and difficult to read.
- **Normalisation choices:** The [0, 1] normalisation we use shows relative ranking within the peer group.  A firm that scores 1.0 on a spoke is the best among peers, not necessarily "good" in absolute terms.

> **CFA Exam Tip:** While radar charts are not directly tested on the CFA exam, the concept of multi-dimensional comparison is central to the curriculum.  Be comfortable interpreting any visualisation that presents multiple ratios simultaneously.

#### Constructing the Radar Chart

The construction process involves several steps:

1. **Select ratios:** Choose 5-8 representative ratios spanning multiple families (e.g., ROE, operating margin, asset turnover, current ratio, D/E, interest coverage).
2. **Normalise:** Scale each ratio to [0, 1] using min-max normalisation across the companies being compared.
3. **Handle direction:** For ratios where lower is better (e.g., D/E), invert the normalisation so that the "best" value maps to 1.0.
4. **Plot:** Place each ratio on an equally-spaced spoke, connect the points for each company to form a polygon.

> **Common Mistake:** Not inverting "lower is better" ratios.  If D/E is not inverted, a highly leveraged firm appears to have a larger polygon, which falsely suggests better performance.  Always ensure all spokes point outward in the "good" direction.

In [ ]:
# ---------------------------------------------------------------
# 10. Radar (Spider) Chart — Cross-Sectional Comparison (Year 5)
# ---------------------------------------------------------------

def compute_summary_ratios(c, idx=-1):
    """Return a dict of key ratios for a single year."""
    return {
        "Net Margin":        c["net_income"][idx] / c["revenue"][idx],
        "ROE":               c["net_income"][idx] / c["equity"][idx],
        "Asset Turnover":    c["revenue"][idx] / c["total_assets"][idx],
        "Current Ratio":     c["current_assets"][idx] / c["current_liab"][idx],
        "Interest Coverage": c["operating_income"][idx] / c["interest_expense"][idx],
        "D/E":               c["total_liab"][idx] / c["equity"][idx],
    }

# Gather data
ratio_names = list(compute_summary_ratios(companies[0]).keys())
n_ratios = len(ratio_names)

raw_values = []
for c in companies:
    vals = compute_summary_ratios(c)
    raw_values.append([vals[r] for r in ratio_names])

raw_values = np.array(raw_values)  # (3, n_ratios)

# Normalise to [0, 1] across companies for each ratio
# For D/E, lower is "better", so invert
invert = [False, False, False, False, False, True]  # D/E inverted
normed = np.zeros_like(raw_values)
for j in range(n_ratios):
    col = raw_values[:, j]
    if invert[j]:
        col = -col  # invert so higher = better
    mn, mx = col.min(), col.max()
    if mx - mn > 1e-12:
        normed[:, j] = (col - mn) / (mx - mn)
    else:
        normed[:, j] = 0.5

# Radar chart
angles = np.linspace(0, 2 * np.pi, n_ratios, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (c, col) in enumerate(zip(companies, colours)):
    values = normed[i].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", color=col, linewidth=2, label=c["name"])
    ax.fill(angles, values, color=col, alpha=0.1)

ax.set_thetagrids(np.degrees(angles[:-1]), ratio_names, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title("Radar Chart — Cross-Sectional Comparison (Year 5)\n(normalised to [0,1]; D/E inverted so outward = better)",
             fontsize=12, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

### 11.2 Small-Multiples Visualization

Small-multiples plots show the **5-year trend** for several key ratios side by side, making it easy to spot diverging trends.

The small-multiples approach was popularised by Edward Tufte, who argued that repeated small charts with identical scales are more effective than a single complex chart for revealing patterns and outliers across multiple dimensions.

**Advantages of small-multiples over combined charts:**

| Feature | Combined Chart | Small-Multiples |
|---------|---------------|-----------------|
| Readability | Lines overlap and clutter | Each ratio has its own panel |
| Scale consistency | Must share a single y-axis | Each panel has its own scale |
| Pattern detection | Trends obscured by crossings | Trends clearly visible |
| Company comparison | Colour-coding required | Side-by-side positioning |

> **Key Concept:** The power of small-multiples lies in the viewer's ability to scan across panels quickly, detecting which ratios are improving, which are stable, and which are deteriorating — all at a glance.  The shared time axis (x-axis) ensures temporal alignment across all panels.

When reading the small-multiples chart below, focus on:

1. **Slope direction** — positive slopes indicate improvement, negative slopes indicate deterioration.
2. **Relative levels** — which company line is consistently highest or lowest within each panel.
3. **Volatility** — smooth lines suggest stable performance; jagged lines suggest inconsistency.
4. **Convergence/divergence** — are the companies becoming more similar or more different over time?

## 11. Multi-Company Dashboard

### 11.1 Comprehensive Ratio Table

Below we compile all major ratios into a single table for easy comparison across companies and years.

> **Key Concept:** Dashboards should prioritise *clarity* over *density*.  A well-designed dashboard tells a story; a cluttered one obscures it.

A good financial dashboard follows several design principles:

- **Group related ratios** — keep profitability together, liquidity together, etc.
- **Highlight outliers** — use colour coding to flag ratios that deviate significantly from benchmarks.
- **Show trends** — include arrows or sparklines to indicate direction of change.
- **Provide context** — always show industry medians or peer averages alongside company-specific values.

> **CFA Exam Tip:** On the exam, when presented with a ratio table for multiple companies, start by identifying the outliers — which ratios deviate most from the peer median?  These are the ratios that tell the most interesting story and are most likely to be tested.

#### Dashboard Reading Strategy

When confronted with a comprehensive ratio table, follow this systematic approach:

1. **Scan profitability first:** ROE is the summary metric — start here to identify the leaders and laggards.
2. **Decompose with DuPont:** For the firm with the highest and lowest ROE, check whether it is driven by margins, turnover, or leverage.
3. **Check financial health:** Look at liquidity (current ratio, quick ratio) and solvency (D/E, interest coverage) ratios to assess risk.
4. **Evaluate efficiency:** Examine activity ratios to understand how well assets are deployed.
5. **Consider valuation:** If market data is available, check whether the market rewards or penalises the observed performance through valuation multiples.

> **Common Mistake:** Trying to analyse every ratio in the table.  In practice, focus on the 5-7 most important ratios for the specific analytical question you are trying to answer.  For a credit analysis, emphasise solvency and liquidity.  For an equity analysis, emphasise profitability and valuation.

In [ ]:
# ---------------------------------------------------------------
# 11.1 Comprehensive Ratio Comparison Table (Year 5)
# ---------------------------------------------------------------

def all_ratios_year(c, idx=-1):
    """Return an ordered dict of all ratios for a single year."""
    rev = c["revenue"][idx]
    ni  = c["net_income"][idx]
    ta  = c["total_assets"][idx]
    eq  = c["equity"][idx]
    eps = ni / c["shares_outstanding"][idx]
    bvps = eq / c["shares_outstanding"][idx]
    ev = c["market_cap"][idx] + c["long_term_debt"][idx] - c["cash"][idx]

    ratios = {}
    # Profitability
    ratios["Gross Margin"]      = c["gross_profit"][idx] / rev
    ratios["Operating Margin"]  = c["operating_income"][idx] / rev
    ratios["Net Margin"]        = ni / rev
    ratios["ROA"]               = ni / ta
    ratios["ROE"]               = ni / eq
    # Activity
    ratios["Receivables TO"]    = rev / c["receivables"][idx]
    ratios["Inventory TO"]      = c["cogs"][idx] / c["inventory"][idx]
    ratios["Total Asset TO"]    = rev / ta
    ratios["Fixed Asset TO"]    = rev / c["ppe_net"][idx]
    # Liquidity
    ratios["Current Ratio"]     = c["current_assets"][idx] / c["current_liab"][idx]
    ratios["Quick Ratio"]       = (c["cash"][idx] + c["receivables"][idx]) / c["current_liab"][idx]
    ratios["Cash Ratio"]        = c["cash"][idx] / c["current_liab"][idx]
    # Solvency
    ratios["D/E"]               = c["total_liab"][idx] / eq
    ratios["D/A"]               = c["total_liab"][idx] / ta
    ratios["Interest Coverage"] = c["operating_income"][idx] / c["interest_expense"][idx]
    # Valuation
    ratios["P/E"]               = c["share_price"][idx] / eps
    ratios["P/B"]               = c["share_price"][idx] / bvps
    ratios["EV/EBITDA"]         = ev / c["ebitda"][idx]
    return ratios

# Print table
header_names = list(all_ratios_year(companies[0]).keys())
print(f"{'Ratio':<22}", end="")
for c in companies:
    print(f" {c['name']:>12}", end="")
print()
print("-" * 60)

for rname in header_names:
    print(f"{rname:<22}", end="")
    for c in companies:
        val = all_ratios_year(c)[rname]
        # Format percentages vs ratios
        if rname in ["Gross Margin", "Operating Margin", "Net Margin", "ROA", "ROE", "D/A"]:
            print(f" {val:>11.1%}", end="")
        else:
            print(f" {val:>11.2f}", end="")
    print()

#### Interpretation: Comprehensive Ratio Table

With all ratios compiled in a single table, several patterns emerge:

1. **Cross-sectional outliers:** Identify which company ranks highest and lowest for each ratio.  Consistent top performance across profitability ratios signals genuine operational excellence, not just accounting artefacts.
2. **Internal consistency checks:** Ratios should tell a coherent story.  If a firm has high ROE but low interest coverage, the ROE is likely leverage-driven — a finding confirmed by the DuPont decomposition.
3. **Red flags:** Watch for combinations like rising revenue + falling margins (pricing pressure), improving profitability + deteriorating liquidity (earnings quality concerns), or increasing leverage + declining coverage (growing default risk).

> **Common Mistake:** Focusing on a single ratio in isolation.  Financial analysis is inherently multi-dimensional — a firm's overall health cannot be captured by any single metric.  Always examine ratios in families and look for consistency across families.

#### Ratio Correlation Patterns

Certain ratio movements tend to occur together:

| If You See... | Also Check... | Because... |
|---------------|--------------|------------|
| Rising ROE | D/E trend | ROE increase may be leverage-driven |
| Falling gross margin | Inventory turnover | Rising COGS may signal inventory issues |
| Declining current ratio | Cash conversion cycle | Working capital may be tightening |
| Rising D/E | Interest coverage | More debt means more interest burden |
| Improving asset turnover | Margin trend | Margin-turnover trade-off may apply |

> **Key Concept:** The best financial analysts are detectives — they look for inconsistencies between ratios that reveal the true story behind the numbers.  If profitability ratios look strong but cash-flow ratios are weak, the quality of reported earnings deserves scrutiny.

### 11.2 Small-Multiples Visualization

Small-multiples plots show the **5-year trend** for several key ratios side by side, making it easy to spot diverging trends.

The small-multiples approach was popularised by Edward Tufte, who argued that repeated small charts with identical scales are more effective than a single complex chart for revealing patterns and outliers across multiple dimensions.

**Advantages of small-multiples over combined charts:**

| Feature | Combined Chart | Small-Multiples |
|---------|---------------|-----------------|
| Readability | Lines overlap and clutter | Each ratio has its own panel |
| Scale consistency | Must share a single y-axis | Each panel has its own appropriate scale |
| Pattern detection | Trends obscured by crossings | Trends clearly visible in isolation |
| Company comparison | Colour-coding required | Side-by-side positioning with consistent colours |

> **Key Concept:** The power of small-multiples lies in the viewer's ability to scan across panels quickly, detecting which ratios are improving, which are stable, and which are deteriorating — all at a glance.  The shared time axis (x-axis) ensures temporal alignment across all panels.

When reading the small-multiples chart below, focus on:

1. **Slope direction** — positive slopes indicate improvement, negative slopes indicate deterioration.
2. **Relative levels** — which company line is consistently highest or lowest within each panel.
3. **Volatility** — smooth lines suggest stable performance; jagged lines suggest inconsistency.
4. **Convergence/divergence** — are the companies becoming more similar or more different over time?

In [ ]:
# ---------------------------------------------------------------
# 11.2 Small-multiples: 5-year trends for key ratios
# ---------------------------------------------------------------

key_ratios_funcs = {
    "Net Margin (%)":       lambda c: c["net_income"] / c["revenue"] * 100,
    "ROE (%)":              lambda c: c["net_income"] / c["equity"] * 100,
    "Total Asset TO (x)":   lambda c: c["revenue"] / c["total_assets"],
    "Current Ratio (x)":    lambda c: c["current_assets"] / c["current_liab"],
    "D/E (x)":              lambda c: c["total_liab"] / c["equity"],
    "Interest Coverage (x)": lambda c: c["operating_income"] / c["interest_expense"],
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, (title, func) in zip(axes, key_ratios_funcs.items()):
    for c, col in zip(companies, colours):
        ax.plot(years, func(c), marker="o", color=col, linewidth=2, label=c["name"])
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks(years)
    ax.set_xlabel("Year")

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Multi-Company Dashboard — 5-Year Trends", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

#### Interpretation: Small-Multiples Trends

The small-multiples visualisation reveals divergent trajectories across the three companies:

- **Profitability trends:** Look for firms where margins are expanding (positive slope) versus compressing (negative slope).  Margin compression often precedes earnings disappointments.
- **Leverage trends:** Increasing debt-to-equity over time signals growing financial risk, especially if not accompanied by corresponding growth in profitability.
- **Turnover trends:** Declining asset turnover may indicate the firm is investing in assets that have not yet begun generating revenue (positive if growth capex) or that existing assets are becoming less productive (negative if operational inefficiency).

> **CFA Exam Tip:** When analysing trend charts, always identify the *inflection points* — where does the trend change direction?  These inflection points often correspond to strategic decisions (acquisitions, divestitures, pricing changes) or external shocks (recessions, regulatory changes).

#### How to Narrate a Small-Multiples Chart

When presenting small-multiples analysis (on the exam or in practice), structure your narrative as follows:

1. **Overall direction:** "Over the five-year period, AlphaCorp's profitability ratios show a steady upward trend while GammaTech's are flat."
2. **Relative positioning:** "BetaInc consistently leads on margin metrics but lags on asset turnover, consistent with its capital-intensive growth strategy."
3. **Convergence or divergence:** "The three companies' leverage ratios are converging toward a common level, suggesting industry capital-structure norms are exerting gravitational pull."
4. **Anomalies:** "GammaTech shows a sharp drop in interest coverage in Year 3, likely due to a debt issuance that year — worth investigating further."

> **Key Concept:** Small-multiples charts are one of the most effective visualisation formats for time-series comparison because they avoid the clutter of overlapping lines on a single chart while maintaining visual comparability through shared scales and alignment.

#### Identifying Convergence and Divergence

One of the most valuable insights from small-multiples analysis is identifying whether companies' ratios are **converging** (becoming more similar) or **diverging** (becoming more different) over time:

- **Convergence** suggests competitive equilibrium — firms are adopting similar strategies and achieving similar results.  This is common in mature, competitive industries.
- **Divergence** suggests differentiation — one firm is pulling ahead (or falling behind) its peers.  This is the most actionable finding for investors, as it may signal an emerging competitive advantage or disadvantage.

> **Key Concept:** Diverging trends are more interesting than converging trends from an investment perspective.  A firm whose profitability ratios are diverging upward from peers may represent an emerging competitive moat — exactly the kind of advantage that drives long-term stock outperformance.

### 11.3 DuPont Factor Decomposition — Stacked View

The chart below shows how each DuPont factor contributes to ROE across companies and years, providing a unified view of the decomposition dynamics.

> **CFA Exam Tip:** When asked to "explain the change in ROE," always decompose it.  Simply stating "ROE went up" without attribution to margin, turnover, or leverage will not earn full marks.

#### Reading a Stacked DuPont Chart

In a stacked visualisation of DuPont factors:

- **The total height** of each bar represents the overall ROE (or a log-transformed version).
- **Each segment's height** represents one factor's contribution to the total.
- **Changes in segment height** over time reveal which factor is driving ROE changes.
- **Relative segment proportions** across companies reveal different business models.

This view is particularly powerful for identifying the *source* of ROE changes.  If the total bar height increases but only the leverage segment grows, the ROE improvement is entirely debt-driven — a concerning pattern that warrants deeper investigation.

> **Key Concept:** The stacked DuPont chart is the ultimate diagnostic tool for ROE.  It simultaneously shows the level, the composition, and the trend of ROE for each company — all in a single visual.  Mastering its interpretation is essential for financial statement analysis.

#### What to Look For in the Stacked View

| Pattern | Interpretation | Investment Implication |
|---------|---------------|----------------------|
| Margin segment dominates | Profitability-driven ROE | High quality, sustainable |
| Leverage segment dominates | Leverage-driven ROE | Lower quality, higher risk |
| Even distribution | Balanced contributors | Moderate quality |
| Margin segment growing over time | Improving operations | Positive signal |
| Leverage segment growing over time | Increasing financial risk | Negative signal — monitor solvency |

In [ ]:
# ---------------------------------------------------------------
# 11.3 DuPont 3-factor stacked bar chart (all companies, all years)
# ---------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, c, col in zip(axes, companies, [PRIMARY, SECONDARY, TERTIARY]):
    npm, ato, em, roe = dupont_3(c)

    # For stacked bar, show each factor's value (scaled for visibility)
    x = np.arange(n_years)
    w = 0.55

    ax.bar(x, npm * 100, w, label="Net Profit Margin (%)", color=PRIMARY, alpha=0.85)
    ax.bar(x, ato * 10, w, bottom=npm * 100, label="Asset Turnover (x10)", color=SECONDARY, alpha=0.85)
    ax.bar(x, em, w, bottom=npm * 100 + ato * 10, label="Equity Multiplier (x1)", color=TERTIARY, alpha=0.85)

    # Annotate ROE
    for i in range(n_years):
        total_h = npm[i] * 100 + ato[i] * 10 + em[i]
        ax.text(i, total_h + 0.3, f"ROE={roe[i]:.1%}", ha="center", fontsize=8, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([f"Yr {y}" for y in years])
    ax.set_title(c["name"], fontsize=12, fontweight="bold")

axes[0].set_ylabel("Factor Value (scaled)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("DuPont 3-Factor Components (Scaled for Visualisation)", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

#### Interpretation: DuPont Stacked View

The stacked bar chart above provides a powerful visual summary of how each DuPont factor contributes to ROE.  Key observations:

- **AlphaCorp** shows a balanced profile — moderate contributions from each factor with slight leverage tilt.
- **BetaInc** derives most of its ROE from strong operating margins, with leverage playing a minimal role.  This is the "quality" ROE profile that equity investors prefer.
- **GammaTech** relies heavily on leverage (a tall equity-multiplier segment) to achieve competitive ROE.  While effective, this strategy leaves little margin for error if operating performance weakens.

> **Key Concept:** The ideal ROE composition depends on the firm's industry and risk tolerance.  High-margin/low-leverage ROE is generally preferred because it is more sustainable and less vulnerable to interest-rate shocks or credit-market disruptions.

#### Quality of ROE Assessment

Analysts often classify ROE quality on a spectrum:

| ROE Source | Quality | Sustainability | Risk |
|------------|---------|---------------|------|
| High operating margin | Highest | Most sustainable (reflects competitive advantage) | Low |
| High asset turnover | High | Sustainable if operational efficiency is maintained | Low-Moderate |
| Moderate leverage | Moderate | Sustainable at stable interest rates | Moderate |
| Excessive leverage | Low | Vulnerable to interest-rate changes, credit-market disruption | High |

> **CFA Exam Tip:** When evaluating two companies with similar ROEs, always prefer the one whose ROE is driven by operational factors (margin and turnover) rather than leverage.  This is a common exam question format: "Which company has higher-quality ROE and why?"

A practical rule of thumb: if the equity multiplier exceeds 3.0x and is the dominant contributor to ROE, flag the company for detailed solvency analysis before concluding that its ROE is attractive.

## Synthesis: Putting It All Together

Before we summarise, let us reflect on the analytical framework we have built throughout this notebook.

### The Complete Analytical Process

A comprehensive financial ratio analysis follows this sequence:

| Step | Action | Tools Used |
|------|--------|-----------|
| 1 | Gather financial statements | 10-K filings, data providers |
| 2 | Adjust for accounting differences | FIFO/LIFO conversion, lease capitalisation |
| 3 | Compute all five ratio families | Formulas from Sections 3-7 |
| 4 | Perform DuPont decomposition | 3-factor and 5-factor (Sections 8-9) |
| 5 | Cross-sectional comparison | Peer benchmarks, radar charts (Section 10) |
| 6 | Time-series trend analysis | Small-multiples, trend lines (Section 10) |
| 7 | Synthesise findings | Dashboard, narrative (Section 11) |
| 8 | Form conclusions | Investment thesis, credit assessment |

### Common Analytical Patterns

Through extensive practice with ratio analysis, analysts develop pattern recognition for common financial profiles:

**The "Quality Compounder":**
- High and stable ROE (15-25%) driven primarily by margins and turnover, not leverage.
- Consistent or expanding operating margins over time.
- Low D/E ratio with strong interest coverage.
- This is the profile most sought by long-term equity investors (e.g., Berkshire Hathaway's investment targets).

**The "Leveraged Operator":**
- High ROE (20%+) driven primarily by the equity multiplier.
- Moderate or thin operating margins.
- High D/E with coverage ratios near covenant thresholds.
- This profile is common in private equity-owned firms and carries significant downside risk.

**The "Turnaround Candidate":**
- Low or declining ROE, but with identifiable drivers through DuPont analysis.
- Improving operational metrics (margins, turnover) masked by legacy leverage issues.
- Current ratio and liquidity metrics stabilising or improving.
- This profile attracts activist investors and special-situation funds.

**The "Cash-Rich Underleveraged":**
- Moderate ROE despite strong operating performance.
- Very low equity multiplier (minimal debt, large cash holdings).
- High current ratio and cash ratio.
- This firm could increase ROE through leverage or share buybacks — a potential catalyst for activist involvement.

> **Key Concept:** The ability to recognise these patterns quickly is what separates novice analysts from experienced ones.  The DuPont framework is the essential diagnostic tool that enables this pattern recognition.

> **CFA Exam Tip:** The CFA exam often presents a case study requiring you to analyse a firm's financial statements, compute key ratios, decompose ROE, and form an overall assessment.  Practice the complete analytical sequence — from raw data to conclusion — as a unified process, not as isolated ratio calculations.

## 12. Summary & Key Takeaways

> **Key Concept:** Ratio analysis transforms raw financial statements into comparable, interpretable metrics. The DuPont framework is the crown jewel — it tells you not just *what* ROE is, but *why* it is what it is.

**What we covered:**

1. **Five ratio families** — profitability, activity, liquidity, solvency, and valuation — each answering a different question about company health.
2. **DuPont 3-factor decomposition** — ROE = Net Margin × Asset Turnover × Leverage.
3. **DuPont 5-factor decomposition** — further separating tax burden, interest burden, and operating margin.
4. **Cross-sectional and time-series analysis** — two complementary lenses for interpreting ratios.
5. **Visualisation techniques** — radar charts, tornado charts, small-multiples dashboards.

### Key Formulas Reference Table

| Ratio | Formula | Family |
|-------|---------|--------|
| Gross Margin | (Revenue - COGS) / Revenue | Profitability |
| Operating Margin | EBIT / Revenue | Profitability |
| Net Margin | Net Income / Revenue | Profitability |
| ROA | Net Income / Total Assets | Profitability |
| ROE | Net Income / Equity | Profitability |
| Receivables Turnover | Revenue / Accounts Receivable | Activity |
| Inventory Turnover | COGS / Inventory | Activity |
| Total Asset Turnover | Revenue / Total Assets | Activity |
| Fixed Asset Turnover | Revenue / Net PP&E | Activity |
| Payables Turnover | COGS / Accounts Payable | Activity |
| Current Ratio | Current Assets / Current Liabilities | Liquidity |
| Quick Ratio | (Cash + Receivables) / Current Liabilities | Liquidity |
| Cash Ratio | Cash / Current Liabilities | Liquidity |
| D/E Ratio | Total Liabilities / Equity | Solvency |
| D/A Ratio | Total Liabilities / Total Assets | Solvency |
| Interest Coverage | EBIT / Interest Expense | Solvency |
| Fixed Charge Coverage | (EBIT + Lease) / (Interest + Lease) | Solvency |
| P/E | Price per Share / EPS | Valuation |
| P/B | Price per Share / Book Value per Share | Valuation |
| P/S | Market Cap / Revenue | Valuation |
| EV/EBITDA | Enterprise Value / EBITDA | Valuation |

### DuPont Decompositions

**3-Factor:**

$$
\text{ROE} = \frac{\text{NI}}{\text{Rev}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

**5-Factor:**

$$
\text{ROE} = \frac{\text{NI}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Rev}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

### CFA Exam Preparation Notes

> **CFA Exam Tip:** The most commonly tested topics in this area are:
>
> 1. **Computing and interpreting** all five ratio families from given financial statements.
> 2. **Performing a DuPont decomposition** (both 3-factor and 5-factor) and explaining what each component reveals.
> 3. **Identifying the driver** of a change in ROE — was it margin improvement, better asset utilisation, or increased leverage?
> 4. **Limitations of ratio analysis** — accounting differences, industry comparability, seasonality, window dressing.
> 5. **Cross-sectional vs. time-series analysis** — when to use each and what pitfalls to avoid.

> **Common Mistake:** Memorising formulas without understanding interpretation.  The exam rarely asks you to simply compute a ratio — it asks you to *interpret* the result, *compare* it to benchmarks, and *explain* what it implies about the firm's strategy, risk, or performance trajectory.

---

*End of notebook.*